# Prompt Engineering y Structured Outputs — Notebook (Clase 7)

Este notebook es el **recurso central** de la Semana 7. La idea es **mostrar primero y explicar después con las slides**: cada sección corre un demo que produce una observación, y el deck explica el *porqué*. Al final de cada sección hay un puntero ▶ a las láminas que la explican.

El hilo de la clase: **dirigir la distribución del próximo token sin tocar los pesos**. Sobre ese eje, dos palancas opuestas sobre la misma distribución —**expandir** el cómputo (razonamiento) y **restringir** la salida (structured outputs)— y un escalón meta (automatizar y situar el prompt). Y atrás de todo, la pregunta de la Clase 6: ¿se moldea la **forma** o la **capacidad**?

**Cómo está armado.** Todo corre sobre Ollama + Qwen3 (local, sin API key ni costo) y HuggingFace para las dos demos a nivel de logits. El instrumento que atraviesa el notebook es un **harness de evaluación** (portado del framework del curso de Claude): de acá en más, cada afirmación del deck se **mide**, no se postula.

> **Regla del día:** *ver, no postular*. Corré cada celda, mirá la salida, y recién después mirá la lámina.

## 0 · Preparación

Levantamos **Qwen3** en local con Ollama y cargamos las librerías open-source. Nada de APIs pagas.

> **Activá la GPU:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU.*


In [8]:
import sys, platform, os
print(sys.platform, platform.platform(), os.getcwd())


linux Linux-6.6.122+-x86_64-with-glibc2.39 /content


In [9]:
# ¿Hay GPU? Si no lista ninguna, activala (Entorno de ejecución -> T4 GPU).
!nvidia-smi

Tue Sep 15 17:08:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
# zstd es necesario para descomprimir el instalador de Ollama en Colab.
!apt-get -qq update
!apt-get -qq install -y zstd

# Instala Ollama (detecta la GPU automáticamente).
!curl -fsSL https://ollama.com/install.sh | sh

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [11]:
import subprocess, time, requests

def ollama_vivo():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        return True
    except requests.exceptions.RequestException:
        return False

# Guard de reentrada: si el server ya responde, no lanzamos un segundo proceso
# (el segundo no puede bindear el puerto y muere, pero ensucia la salida).
if ollama_vivo():
    print("Ollama ya estaba corriendo.")
else:
    # stdout/stderr al vacío: si no, los logs del server heredan el stdout del kernel
    # y se filtran en el output de celdas posteriores, en momentos arbitrarios.
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if ollama_vivo():
            print("Ollama está listo.")
            break
        time.sleep(1)
    else:
        print("El servidor tardó en levantar; volvé a correr esta celda.")

Ollama ya estaba corriendo.


In [12]:
# Usamos CUATRO modelos, a propósito:
#  - MODEL: la variante *instruct* (4B), que responde directo. Default de casi todo.
#  - MODEL_THINK: la variante de *razonamiento* (4B). Sección 8.
#  - MODEL_SMALL y MODEL_TINY: dos escalas por debajo. No es capricho de performance:
#    varias técnicas sólo muestran su efecto donde el modelo tiene dónde fallar. Y con
#    TRES puntos en la curva se ve algo que con dos no se ve: abajo de cierto tamaño las
#    técnicas no sólo dejan de ayudar, empiezan a hacer daño (Sección 3).
MODEL       = "qwen3:4b-instruct"
MODEL_THINK = "qwen3:4b"

import subprocess

def pull(m, obligatorio=True):
    print("pull", m, "...", flush=True)
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    if r.returncode == 0:
        return True
    if obligatorio:
        raise RuntimeError(
            "Falló 'ollama pull " + m + "'.\n" + (r.stderr or r.stdout).strip() +
            "\n\nRevisá que el tag exista en https://ollama.com/library/qwen3/tags"
            "\n(ojo: las variantes 2507 llevan sufijo, p.ej. qwen3:4b-instruct-2507)."
        )
    print("   no disponible, sigo con el próximo candidato")
    return False

pull(MODEL)
pull(MODEL_THINK)

def primero_disponible(candidatos, etiqueta):
    for c in candidatos:
        if pull(c, obligatorio=False):
            return c
    raise RuntimeError("Ningún modelo disponible para " + etiqueta + ": " + str(candidatos))

MODEL_SMALL = primero_disponible(["qwen2.5:1.5b-instruct", "qwen2.5:3b-instruct",
                                  "llama3.2:1b"], "MODEL_SMALL")
MODEL_TINY  = primero_disponible(["qwen2.5:0.5b-instruct", "qwen2.5:1.5b-instruct"],
                                 "MODEL_TINY")

print("\nEscalas:", MODEL_TINY, "<", MODEL_SMALL, "<", MODEL, "\n")
!ollama list

pull qwen3:4b-instruct ...
pull qwen3:4b ...
pull qwen2.5:1.5b-instruct ...
pull qwen2.5:0.5b-instruct ...

Escalas: qwen2.5:0.5b-instruct < qwen2.5:1.5b-instruct < qwen3:4b-instruct 

NAME                     ID              SIZE      MODIFIED               
qwen2.5:1.5b-instruct    65ec06548149    986 MB    Less than a second ago    
qwen2.5:0.5b-instruct    a8b0c5157701    397 MB    Less than a second ago    
qwen3:4b-instruct        0edcdef34593    2.5 GB    1 second ago              
qwen3:4b                 359d7dd4bcda    2.5 GB    1 second ago              


In [13]:
# Librerías open-source. Nada de APIs pagas.
#
# OJO: en Colab, torch / torchvision / transformers / numpy YA vienen instalados y
# APAREADOS entre sí. NO agregues "-U torch": actualiza torch y deja torchvision
# desapareado; transformers importa torchvision de forma lazy y la Sección 11 muere con
# "RuntimeError: operator torchvision::nms does not exist". Lo único que falta es ollama.
!pip -q install ollama pydantic

import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
# El warning de HF_TOKEN sale por warnings.warn Y por el logger de huggingface_hub:
# transformers.logging no lo alcanza. Hay que callar los dos.
warnings.filterwarnings("ignore", module="huggingface_hub")
# setLevel en el padre NO alcanza: los loggers hijos de huggingface_hub traen su
# propio nivel, así que el warning de "unauthenticated requests" sobrevivía.
for _n in list(logging.root.manager.loggerDict):
    if _n.startswith("huggingface_hub"):
        logging.getLogger(_n).setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

import ollama, torch, numpy as np, transformers
from pydantic import BaseModel, Field
from torchvision.ops import nms   # canario: si el par torch/torchvision está roto, falla
                                  # ACÁ (celda 6) y no a mitad de clase en la Sección 11.
transformers.logging.set_verbosity_error()

# SMOKE_TEST=True: corrida rápida para dar la clase. Controla cuánto cómputo gastamos,
# NO el tamaño de los sets de evaluación (ya son chicos; achicarlos sólo agregaría ruido).
SMOKE_TEST = True

N_CHAINS = 5    if SMOKE_TEST else 9        # Sec. 7: cadenas por problema
NP_COT   = 1024 if SMOKE_TEST else 2048     # Sec. 5-7: presupuesto de la cadena
NP_DIR   = 16                               # Sec. 5/8: respuesta directa = sólo el número
N_APE    = 3    if SMOKE_TEST else 5        # Sec. 16: candidatos de APE

# Escalas. La Sección 3 usa las TRES (ahí es donde la curva completa dice algo);
# las Secciones 7, 13b y 15 usan dos, con el extremo que les sirve.
ESCALAS3 = [("diminuto", MODEL_TINY), ("chico", MODEL_SMALL), ("4B", MODEL)]
ESCALAS  = [("diminuto", MODEL_TINY), ("4B", MODEL)]

# Cada sección deja acá sus números medidos. La tabla de cierre (Sección 17) se GENERA
# desde este dict: así el resumen no puede contradecir a lo que se vio en pantalla.
RESULTADOS = {}

torch.manual_seed(1337)
np.random.seed(1337)
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("Modelos -> 4B:", MODEL, "| razonador:", MODEL_THINK)
print("           chico:", MODEL_SMALL, "| diminuto:", MODEL_TINY)
print("SMOKE_TEST:", SMOKE_TEST)

torch 2.11.0+cu128 | transformers 5.16.1
Modelos -> 4B: qwen3:4b-instruct | razonador: qwen3:4b
           chico: qwen2.5:1.5b-instruct | diminuto: qwen2.5:0.5b-instruct
SMOKE_TEST: True


**Un ayudante que reusamos en todo el notebook.** `ask` envuelve `ollama.chat` sobre el modelo *instruct* (`MODEL`), que responde directo. Devuelve el texto pelado de `.content`. Dos agregados respecto de otras clases: `fmt`, que —cuando se lo pasás— activa **structured outputs** (lo vemos desde la lámina 19; acá el juez ya lo usa para devolver un score parseable), y `num_predict`, que topea cuántos tokens genera (así una respuesta no se va de mambo).

> **Decisión didáctica, y se avisa:** Qwen3 separa el razonamiento (`message.thinking`) de la respuesta (`message.content`). El modelo *instruct* no razona, así que `.thinking` viene vacío y `.content` sale limpio. Cuando en la Sección 8 usemos el modelo de razonamiento con `think=True`, el monólogo va a `.thinking` y lo podemos inspeccionar con `return_thinking=True`. `think` y `format` son parámetros de nivel superior (no van dentro de `options`).


In [14]:
def ask(prompt, system=None, temperature=0.0, model=MODEL, think=False,
        fmt=None, num_predict=1024, return_thinking=False, avisar=True):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    opts = {"temperature": temperature, "num_predict": num_predict}
    kwargs = dict(model=model, messages=msgs, think=think, options=opts)
    if fmt is not None:
        kwargs["format"] = fmt          # structured outputs (constrained decoding)
    r = ollama.chat(**kwargs)

    # ¿El modelo TERMINÓ, o lo cortamos nosotros? done_reason == "length" significa que
    # se acabó el presupuesto de tokens. Esto importa mucho más de lo que parece: una
    # respuesta cortada se lee como una respuesta mala, y el juez la penaliza como tal.
    # Sin esta marca, estaríamos midiendo num_predict y llamándolo calidad del prompt.
    if avisar and getattr(r, "done_reason", None) == "length":
        print("   [⚠ cortado por num_predict=" + str(num_predict) + "]")

    content = (r.message.content or "").strip()
    # Fallback: si algún build mete el razonamiento en .content cerrado con </think>,
    # nos quedamos con lo de después. (Con el instruct no hace falta; con .thinking, tampoco.)
    if "</think>" in content:
        content = content.split("</think>")[-1].strip()
    if return_thinking:
        return content, (getattr(r.message, "thinking", None) or "")
    return content

# ---------------------------------------------------------------------------
# Sanity checks: verificamos la mecánica que importa antes de arrancar.
# OJO: la llamada 1) tarda ~90 s. No está colgado: Ollama está cargando el modelo
# en la GPU por primera vez. Las siguientes son de segundos.
# ---------------------------------------------------------------------------
import time

# 1) El instruct responde, rápido y en una palabra.
t = time.time()
r1 = ask("Respondé en una sola palabra: ¿capital de Francia?", num_predict=16, avisar=False)
print("1) instruct .........:", repr(r1), "|", round(time.time() - t, 1), "s (incluye la carga del modelo)")

# 2) El instruct NO razona: .thinking vacío y .content limpio.
resp, pensa = ask("¿Cuánto es 17 + 25? Respondé solo el número.", return_thinking=True)
print("2) instruct .thinking:", "vacío OK" if not pensa else "OJO no vacío", "| resp:", repr(resp))

# 3) El modelo de razonamiento SÍ separa: monólogo -> .thinking, respuesta -> .content.
t = time.time()
resp2, pensa2 = ask("¿Cuánto es 17 + 25? Al final escribí 'Respuesta: <número>'.",
                    model=MODEL_THINK, think=True, num_predict=2048, return_thinking=True)
print("3) razonador ........:", repr(resp2), "| .thinking:", len(pensa2), "chars |",
      round(time.time() - t, 1), "s")

# 4) Structured outputs (lo que usa el juez por dentro): la salida cumple el schema.
class _Chk(BaseModel):
    ciudad: str
raw = ask("Devolveme la capital de Francia como JSON.", fmt=_Chk.model_json_schema())
try:
    ok = bool(_Chk.model_validate_json(raw))
except Exception:
    ok = False
print("4) structured output :", repr(raw), "| parsea OK:", ok)

# 5) El modelo chico también responde.
print("5) modelo chico .....:", repr(ask("Respondé en una sola palabra: ¿capital de Italia?",
                                         model=MODEL_SMALL, num_predict=16, avisar=False)))

1) instruct .........: 'París' | 3.8 s (incluye la carga del modelo)
2) instruct .thinking: vacío OK | resp: '42'
3) razonador ........: 'Respuesta: 42' | .thinking: 612 chars | 13.5 s
4) structured output : '{\n  "ciudad": "París"\n}' | parsea OK: True
5) modelo chico .....: 'Milán'


**Un ayudante para mirar de cerca.** De acá en más, antes de cada medición vamos a imprimir —para una o dos entradas— **el prompt exacto que se armó y la respuesta cruda del modelo**, así se ve el artefacto concreto antes del promedio. `mostrar_ejemplo` es sólo para eso.


In [15]:
def mostrar_ejemplo(titulo, prompt, respuesta):
    print("─" * 70)
    print(titulo)
    print("─" * 70)
    print("PROMPT:")
    print(prompt)
    print("\nRESPUESTA DEL MODELO:")
    print(str(respuesta))
    print()

def tabla(titulo, columnas, filas):
    """Imprime una tabla chiquita. filas = [(nombre, {columna: valor}), ...].

    La usamos cada vez que comparamos las DOS ESCALAS (modelo chico vs 4B), que es el
    patrón nuevo de las Secciones 3, 7 y 15."""
    w0 = max([len(str(f[0])) for f in filas] + [10]) + 2
    w = max([len(str(c)) for c in columnas] + [8]) + 3
    print("\n" + titulo)
    print("─" * (w0 + w * len(columnas)))
    print("".ljust(w0) + "".join(str(c).rjust(w) for c in columnas))
    for nombre, vals in filas:
        print(str(nombre).ljust(w0) + "".join(str(vals.get(c, "—")).rjust(w) for c in columnas))
    print()

## 1 · Mismo modelo, salidas distintas

El punto de partida de toda la clase: un **mismo checkpoint**, los mismos pesos congelados, produce salidas radicalmente distintas según cómo le pedimos las cosas. No es magia — es que el prompt **es** la condición de una distribución que ya está entrenada.

Miralo: mismo modelo, dos prompts, dos salidas.


In [16]:
prompt_vago      = "Contame sobre las abejas."
prompt_especifico = ("Sos un divulgador científico. En exactamente 3 bullets, explicá "
                     "cómo se comunican las abejas. Cada bullet, máximo 12 palabras.")

# num_predict amplio: queremos ver la respuesta vaga ENTERA, no un corte a mitad de frase.
print("=== PROMPT VAGO ===")
r_vago_demo = ask(prompt_vago, num_predict=1500)
print(r_vago_demo, "\n")
print("=== PROMPT ESPECÍFICO ===")
r_esp_demo = ask(prompt_especifico)
print(r_esp_demo)

print("\nLargo: vago =", len(r_vago_demo), "caracteres | específico =", len(r_esp_demo))

=== PROMPT VAGO ===
¡Claro! Las abejas son una de las especies más fascinantes y útiles en el mundo natural. Aquí te cuento sobre ellas de forma clara y detallada:

---

### 🐝 ¿Qué son las abejas?

Las abejas son insectos pertenecientes a la familia **Apidae**, y son conocidas principalmente por su papel clave en la **polinización** de plantas. Existen diferentes tipos de abejas, pero las más conocidas son las **abejas obreras**, **abejas reina** y **abejas zánganos**.

---

### 🏷️ Estructura social de una colmena

Una colmena (colmena de abejas) es una comunidad social muy organizada, compuesta por:

1. **La reina**  
   - Es la única hembra que se reproduce (puede tener hasta 10.000 huevos por día).  
   - Su función es poner huevos y, en algunos casos, producir jaleas (una sustancia que se usa para alimentar a las crías).  
   - Vive mucho tiempo (de 2 a 5 años, dependiendo de la especie).

2. **Las obreras**  
   - Son las abejas más numerosas y trabajadoras.  
   - Tienen funcione

**Lo que hay que fijar.** El modelo no cambió; cambió lo que le entró. Prompt engineering es ingeniería sobre la **condición** de la distribución, sin gradientes ni datos nuevos.

▶ **Láminas 1–3** (el prompt como condición; prompting vs fine-tuning; la hoja de ruta).


## 2 · El instrumento: un harness de evaluación

Antes de tocar ninguna técnica, construimos el **instrumento que las mide**. Es un port del framework de testing del curso de Claude a nuestro stack (Ollama + Qwen3). La idea que funda todo lo que sigue: **un prompt es código — definís éxito, medís, iterás** (lámina 31). Y el juez es otro LLM: es el **LLM-as-judge de la Clase 6** puesto a trabajar.

Para que el notebook quede prolijo, el andamiaje vive en un módulo aparte (`promptlab`) que escribimos e importamos. Lo que **sí** queda a la vista es el helper `ask` (arriba) y el **prompt del juez con su rúbrica** (lo imprimimos abajo), que son el corazón del instrumento.


In [17]:
%%writefile promptlab.py
# Andamiaje del harness de evaluacion. Portado del framework del curso de Claude a Ollama.
# NO es el foco de la clase: es la infra que deja MEDIR cada tecnica. Leelo si te da curiosidad.
import json, re, html
from statistics import mean
from concurrent.futures import ThreadPoolExecutor

def extract_json(text):
    # Encuentra y parsea el primer bloque JSON de un texto. Tolerante a texto alrededor.
    text = (text or "").strip()
    text = re.sub(r"^```(json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    for open_c, close_c in [("[", "]"), ("{", "}")]:
        i, j = text.find(open_c), text.rfind(close_c)
        if i != -1 and j != -1 and j > i:
            try:
                return json.loads(text[i:j + 1])
            except Exception:
                continue
    return None

# Schema para el veredicto del juez: fuerza JSON parseable (structured outputs).
JUDGE_SCHEMA = {
    "type": "object",
    "properties": {"reasoning": {"type": "string"}, "score": {"type": "integer"}},
    "required": ["reasoning", "score"],
}

def dataset_schema(keys):
    # Schema para la GENERACION de casos, armado dinamicamente desde las claves que el
    # prompt bajo test espera. Va envuelto en un objeto ('casos') en vez de un array raiz:
    # es la forma que Ollama acepta de manera mas consistente.
    return {
        "type": "object",
        "properties": {
            "casos": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {k: {"type": "string"} for k in keys},
                    "required": list(keys),
                },
            }
        },
        "required": ["casos"],
    }

def build_judge_prompt(task_description, prompt_inputs, output, extra_criteria):
    # La rubrica del juez. La exponemos para que se vea "el modelo calificando al modelo".
    return (
        "Sos un evaluador estricto. Calificá la SALIDA de un modelo para esta tarea.\n\n"
        "<tarea>\n" + task_description.strip() + "\n</tarea>\n\n"
        "<input>\n" + json.dumps(prompt_inputs, ensure_ascii=False) + "\n</input>\n\n"
        "<salida_del_modelo>\n" + str(output) + "\n</salida_del_modelo>\n\n"
        "<criterios>\n" + extra_criteria.strip() + "\n</criterios>\n\n"
        "Calificá de 0 a 10 (10 = cumple todos los criterios perfectamente).\n"
        "Devolvé un JSON con las claves 'reasoning' (string breve) y 'score' (entero 0-10)."
    )

class PromptEvaluator:
    # Harness: genera casos, corre tu prompt, y puntua con LLM-as-judge.
    # Recibe `ask` (nuestro helper de Ollama) por inyeccion: toda la inferencia pasa por ahi.
    def __init__(self, ask, model=None, max_workers=1):
        self.ask = ask
        self.model = model
        self.max_workers = max_workers
        self.last_results = []

    def _ask(self, prompt, **kw):
        # Si nos pasaron un modelo explicito, lo usamos: eso es lo que hace posible juzgar
        # con un modelo distinto del evaluado. Si no, cae al default de `ask`.
        if self.model:
            kw.setdefault("model", self.model)
        return self.ask(prompt, **kw)

    def generate_dataset(self, task_description, prompt_inputs_spec, num_cases=3):
        # Genera casos de test con el modelo. Usa STRUCTURED OUTPUTS (no "por favor
        # devolveme JSON"): el schema sale de las claves que el prompt bajo test espera.
        keys = list(prompt_inputs_spec.keys())
        spec_txt = "\n".join("- '" + k + "': " + v for k, v in prompt_inputs_spec.items())
        prompt = (
            "Sos un disenador de casos de test. Genera " + str(num_cases) + " casos DIVERSOS "
            "para un prompt que hace esta tarea:\n\n<tarea>\n" + task_description.strip() +
            "\n</tarea>\n\nEl prompt recibe estos inputs:\n" + spec_txt + "\n\n"
            "Cada caso distinto de los otros y resoluble en pocos tokens."
        )
        raw = self._ask(prompt, temperature=1.0, fmt=dataset_schema(keys))
        parsed = extract_json(raw) or {}
        return (parsed.get("casos") or [])[:num_cases]

    def judge(self, task_description, prompt_inputs, output, extra_criteria):
        prompt = build_judge_prompt(task_description, prompt_inputs, output, extra_criteria)
        raw = self._ask(prompt, temperature=0.0, fmt=JUDGE_SCHEMA)  # structured outputs
        parsed = extract_json(raw) or {"reasoning": "(no parseo)", "score": 0}
        try:
            score = int(parsed.get("score", 0))
        except (TypeError, ValueError):
            score = 0
        return {"score": max(0, min(10, score)), "reasoning": parsed.get("reasoning", "")}

    def run_evaluation(self, run_prompt_function, dataset,
                       extra_criteria="", task_description=""):
        def one(case):
            output = run_prompt_function(case)
            verdict = self.judge(task_description, case, output, extra_criteria)
            return {"inputs": case, "output": output, **verdict}
        if self.max_workers and self.max_workers > 1:
            # ex.map preserva el ORDEN del dataset. Con as_completed (que devuelve por
            # orden de finalizacion) results[i] dejaria de corresponder a dataset[i]: los
            # ejemplos que mostramos abajo aparearian el prompt de un caso con la respuesta
            # de otro, sin ningun error visible.
            with ThreadPoolExecutor(max_workers=self.max_workers) as ex:
                results = list(ex.map(one, dataset))
        else:
            results = [one(c) for c in dataset]
        self.last_results = results
        return results

    def summary(self, results=None):
        results = results if results is not None else self.last_results
        scores = [r["score"] for r in results]
        avg = float(mean(scores)) if scores else 0.0   # float: si no, 10 entero se lee raro
        pr = 100 * len([s for s in scores if s >= 7]) / len(scores) if scores else 0
        return {"n": len(results), "avg": round(avg, 2), "pass_rate": round(pr, 1)}

    def report_html(self, results=None):
        results = results if results is not None else self.last_results
        s = self.summary(results)
        rows = ""
        for r in results:
            cls = "hi" if r["score"] >= 7 else ("mid" if r["score"] >= 4 else "lo")
            rows += ("<tr><td><pre>" +
                     html.escape(json.dumps(r["inputs"], ensure_ascii=False)) + "</pre></td>" +
                     "<td><pre>" + html.escape(str(r["output"])[:1200]) + "</pre></td>" +
                     "<td class='" + cls + "'>" + str(r["score"]) + "</td><td>" +
                     html.escape(r["reasoning"]) + "</td></tr>")
        style = ("<style>"
                 ".plq{table-layout:fixed;width:100%;border-collapse:collapse}"
                 ".plq td,.plq th{border:1px solid #ddd;padding:6px;vertical-align:top;"
                 "text-align:left;overflow-wrap:anywhere}"
                 ".plq th{background:#4a4a4a;color:#fff}"
                 ".plq pre{white-space:pre-wrap;overflow-wrap:anywhere;margin:0;font-family:inherit}"
                 ".hi{background:#c8e6c9}.mid{background:#fff3cd}.lo{background:#f8d7da}"
                 "</style>")
        head = ("<h3>Reporte de evaluacion</h3><p><b>Casos:</b> " + str(s["n"]) +
                " &nbsp; <b>Score promedio:</b> " + str(s["avg"]) +
                "/10 &nbsp; <b>Pass rate (>=7):</b> " + str(s["pass_rate"]) + "%</p>")
        cols = ("<colgroup><col style='width:20%'><col style='width:46%'>"
                "<col style='width:7%'><col style='width:27%'></colgroup>")
        return ("<div style='font-family:Arial'>" + head + style +
                "<table class='plq'>" + cols +
                "<tr><th>Input</th><th>Salida</th><th>Score</th>"
                "<th>Razonamiento</th></tr>" + rows + "</table></div>")

Writing promptlab.py


In [18]:
import promptlab
from IPython.display import HTML, display

# El harness recibe `ask` por inyección: toda la inferencia pasa por nuestro helper.
# max_workers=1 -> corrida serial y determinista (lo que queremos en clase). Subirlo es
# seguro: run_evaluation usa ex.map, que preserva el orden del dataset.
evaluator = promptlab.PromptEvaluator(ask=ask, model=MODEL, max_workers=1)
print("promptlab importado. evaluator listo.")

promptlab importado. evaluator listo.


**Miremos la rúbrica del juez.** Esto es lo que hace "el modelo calificando al modelo": le pasamos la tarea, el input, la salida a evaluar y los criterios, y le pedimos un score. Notá que el juez usa `fmt=JUDGE_SCHEMA` por dentro (structured outputs) para que el score siempre venga parseable — es el mismo mecanismo que vamos a diseccionar desde la lámina 19.


In [19]:
ejemplo_rubrica = promptlab.build_judge_prompt(
    task_description="Explicar un concepto de forma clara y breve.",
    prompt_inputs={"tema": "qué es una hipoteca"},
    output="Una hipoteca es un préstamo para comprar una casa que se paga de a poco.",
    extra_criteria="Debe ser correcto, claro y de 1 a 2 oraciones.",
)
print(ejemplo_rubrica)

Sos un evaluador estricto. Calificá la SALIDA de un modelo para esta tarea.

<tarea>
Explicar un concepto de forma clara y breve.
</tarea>

<input>
{"tema": "qué es una hipoteca"}
</input>

<salida_del_modelo>
Una hipoteca es un préstamo para comprar una casa que se paga de a poco.
</salida_del_modelo>

<criterios>
Debe ser correcto, claro y de 1 a 2 oraciones.
</criterios>

Calificá de 0 a 10 (10 = cumple todos los criterios perfectamente).
Devolvé un JSON con las claves 'reasoning' (string breve) y 'score' (entero 0-10).


**Ahora corramos el instrumento entero** sobre un dataset chico y curado (lo dejamos fijo para que sea reproducible; más abajo mostramos cómo el harness también puede *generar* casos solo). El harness corre el prompt sobre cada caso y el juez lo puntúa.


In [20]:
# Dataset curado (fijo, reproducible). Cada caso es un dict con las claves que el prompt espera.
dataset_demo = [
    {"tema": "qué es el interés compuesto"},
    {"tema": "qué es un eclipse solar"},
]

# La función que corre el prompt bajo test. Se ejecuta una vez por caso.
def run_prompt_demo(prompt_inputs):
    p = "Explicá en 1 o 2 oraciones, para alguien de 12 años: " + prompt_inputs["tema"]
    return ask(p)

resultados = evaluator.run_evaluation(
    run_prompt_function=run_prompt_demo,
    dataset=dataset_demo,
    task_description="Explicar un concepto de forma clara y breve para un chico de 12 años.",
    extra_criteria="Correcto, claro, y de 1 a 2 oraciones. Sin jerga.",
)
print("Resumen:", evaluator.summary())
display(HTML(evaluator.report_html()))

Resumen: {'n': 2, 'avg': 10.0, 'pass_rate': 100.0}


Input,Salida,Score,Razonamiento
"{""tema"": ""qué es el interés compuesto""}","El interés compuesto es cuando, además de la cantidad que pusiste al principio, también ganas interés sobre el interés que ya habías ganado. Es como si tu dinero se ""creciera"" cada vez que pasa el tiempo, porque no solo crece por lo que pusiste, sino también por lo que ya ganaste antes.",10,"La explicación es clara, breve y utiliza una metáfora sencilla ('se crece') que un chico de 12 años puede entender. Explica bien el concepto de interés compuesto sin jerga, y en solo dos oraciones. Cumple completamente los criterios."
"{""tema"": ""qué es un eclipse solar""}","Un eclipse solar ocurre cuando la Luna se sitúa entre el Sol y la Tierra, y bloquea parte o todo el Sol, como si fuera un sombrero que cubriera el Sol. Es como si la Luna pasara por delante del Sol y nos tapara su luz, pero solo se ve en lugares específicos del mundo.",10,"La explicación es clara, utiliza una metáfora sencilla ('como si fuera un sombrero') que un chico de 12 años puede entender, y mantiene el contenido preciso sin jerga. Aunque es un poco más larga que dos oraciones, está dentro de lo razonable para una explicación breve y accesible. Cumple bien los criterios de claridad y simplicidad."


**Y el harness también genera sus propios casos.** Hasta acá el dataset lo escribimos nosotros. `generate_dataset` le pide al modelo que invente casos de test diversos para la tarea — y lo hace con **structured outputs**, armando el schema desde las claves que el prompt espera. Es el harness usando la técnica que vamos a ver en la Sección 10: no le *pedimos* JSON, se lo *imponemos* en el decoder.


In [21]:
# El harness genera el dataset solo. Por dentro usa structured outputs (con el schema
# armado desde prompt_inputs_spec), no "por favor devolveme un JSON".
casos_generados = evaluator.generate_dataset(
    task_description="Explicar un concepto de forma clara y breve para un chico de 12 años.",
    prompt_inputs_spec={"tema": "el concepto a explicar, en pocas palabras"},
    num_cases=3,
)
print("Casos generados por el modelo:")
for c in casos_generados:
    print("  -", c)

Casos generados por el modelo:
  - {'tema': "Gravedad del Sol en el Sistema Solar (física básica, 12 años): El Sol no cae porque está en el centro y todo lo que está alrededor lo atrae como un imán enorme, pero no se mueve porque todo lo que gira alrededor está en un camino circular como una rueda que nunca se cae. Si el Sol se fuera, todo se lanzaría por el espacio como una pelota de goma en el aire. Es como si el Sol fuera una fuerza invisible que mantiene a las estrellas y planetas en su lugar, sin tocarlos, pero siempre con un 'empujón' que les dice: 'no salgas'."}
  - {'tema': 'Circuitos eléctricos (electrónica básica): Imagina que un circuito es como una pista de bicicleta: si el cable es un camino y la batería es una fuente de energía, como una pila de chocolate que te da un golpe, entonces el cable debe estar bien conectado, como una cuerda que no se rompe, y el interruptor es como una puerta que abre o cierra. Si la puerta está abierta, el chico no puede pasar. Así, si el inte

**El mensaje de esta sección.** De acá en adelante **medimos, no adivinamos**. Cuando una técnica del deck dice "esto ayuda", la vamos a correr con y sin la técnica y ver moverse el número.

> **Ojo con un detalle:** el juez es un LLM, así que arrastra los límites de la Clase 6 (sesgo de posición, self-preference). No es un oráculo — es una señal ruidosa pero útil. Para tareas con respuesta *verificable* (un número, un JSON válido) vamos a medir **directo**, sin juez: esa distinción (verificable vs juez = RLVR vs RLHF) vuelve en la sección 13.

▶ **Lámina 31** (el prompt como código) y **Clase 6** (LLM-as-judge).


## 3 · Zero, one, few-shot — y qué enseñan los ejemplos

Primera técnica, medida. Sobre clasificación de sentimiento (respuesta **verificable**, así que medimos accuracy directa, sin juez), comparamos cuatro prompts: **zero-shot** (sin ejemplos — en el código, `ejemplos=None`), **one-shot**, **few-shot con labels correctos**, y —el experimento clave de Min et al.— **few-shot con labels barajados**.

La predicción de Min et al.: few-shot le gana a zero-shot, pero correctos vs barajados dan **casi lo mismo**, porque los ejemplos enseñan la *forma* de la tarea (qué labels existen, cómo se responde), no el mapeo correcto.

> **Por qué acá aparecen TRES modelos.** Una técnica no "funciona" o "no funciona" en abstracto: funciona *en un régimen*. Qwen3-4B ya clasifica sentimiento casi perfecto sin ejemplos, así que no le queda margen para que few-shot le aporte nada. Así que corremos el mismo set en tres escalas —**diminuto**, **chico** y **4B**— y miramos la curva completa. Vas a ver tres comportamientos distintos, y el de abajo es el que ningún tutorial cuenta.

Además de la accuracy imprimimos la **distribución de las predicciones**. Ese es el diagnóstico que importa: un modelo puede tener accuracy baja porque se equivoca repartido, o porque contestó **siempre lo mismo**. Son dos patologías completamente distintas y la accuracy sola no las separa.


In [22]:
# Set de evaluación (texto, label correcto). 12 casos y no 6: con 6, la diferencia entre
# 0.83 y 1.00 era UN caso, y cualquier conclusión sobre los labels era ruido de muestreo.
EVAL_SENT = [
    ("La comida estaba fría y el servicio, pésimo.", "negativo"),
    ("¡Me encantó! Volvería mil veces.", "positivo"),
    ("Es un lugar más, ni bueno ni malo.", "neutral"),
    ("Cumple, pero no me sorprendió para nada.", "neutral"),
    ("Una experiencia horrible de principio a fin.", "negativo"),
    ("Increíble atención, todo salió perfecto.", "positivo"),
    ("Abre de lunes a viernes y tiene estacionamiento.", "neutral"),
    ("Nunca más vuelvo: una pérdida de tiempo y de plata.", "negativo"),
    ("Superó todas mis expectativas, lo recomiendo muchísimo.", "positivo"),
    ("Está correcto, hace lo que promete y nada más.", "neutral"),
    ("Me trataron pésimo y encima cobraron de más.", "negativo"),
    ("Una maravilla, salimos felices los cuatro.", "positivo"),
]

# Demostraciones few-shot (distintas del set de evaluación).
SHOTS_OK = [
    ("El producto llegó roto y tarde.", "negativo"),
    ("Excelente calidad, súper recomendable.", "positivo"),
    ("Está bien, nada del otro mundo.", "neutral"),
]
# Mismos TEXTOS, labels rotados: rotar 3 labels distintos garantiza que NINGUNO queda
# con su label correcto. El mapeo texto->label queda roto a propósito (derangement).
_labs = [l for _, l in SHOTS_OK]
SHOTS_RANDOM = [(t, (_labs[1:] + _labs[:1])[i]) for i, (t, _) in enumerate(SHOTS_OK)]

# El prompt que se arma en cada caso: es EXACTAMENTE lo que ve el modelo.
def build_prompt(texto, ejemplos=None):
    shots = ""
    if ejemplos:                              # ejemplos=None -> zero-shot: no se antepone nada
        for t, l in ejemplos:
            shots += "Texto: " + t + "\nSentimiento: " + l + "\n\n"
    return ("Clasificá el sentimiento en UNA sola palabra: positivo, negativo o neutral.\n\n"
            + shots + "Texto: " + texto + "\nSentimiento:")

# Veamos, sobre un texto de muestra, cómo se ve el prompt en cada régimen.
muestra = "El envío tardó un montón, pero el producto es una maravilla."
for nombre, ejs in [("ZERO-SHOT  (0 ejemplos)", None),
                    ("ONE-SHOT   (1 ejemplo)",  SHOTS_OK[:1]),
                    ("FEW-SHOT   (3 ejemplos)", SHOTS_OK)]:
    print("=" * 64)
    print(nombre)
    print("=" * 64)
    print(build_prompt(muestra, ejs))
    print()

ZERO-SHOT  (0 ejemplos)
Clasificá el sentimiento en UNA sola palabra: positivo, negativo o neutral.

Texto: El envío tardó un montón, pero el producto es una maravilla.
Sentimiento:

ONE-SHOT   (1 ejemplo)
Clasificá el sentimiento en UNA sola palabra: positivo, negativo o neutral.

Texto: El producto llegó roto y tarde.
Sentimiento: negativo

Texto: El envío tardó un montón, pero el producto es una maravilla.
Sentimiento:

FEW-SHOT   (3 ejemplos)
Clasificá el sentimiento en UNA sola palabra: positivo, negativo o neutral.

Texto: El producto llegó roto y tarde.
Sentimiento: negativo

Texto: Excelente calidad, súper recomendable.
Sentimiento: positivo

Texto: Está bien, nada del otro mundo.
Sentimiento: neutral

Texto: El envío tardó un montón, pero el producto es una maravilla.
Sentimiento:



**Lo que hay que ver.** Los prompts comparten la misma instrucción y el mismo texto final; lo único que cambia es cuántos pares `Texto → Sentimiento` se anteponen: **cero**, **uno** o **tres**. Eso es todo lo que separa zero-, one- y few-shot — no hay mecanismo especial, es más contexto adelante.

Ahora midamos, en las tres escalas. Sale primero la tabla de accuracy, después la **distribución de predicciones** (el diagnóstico) y al final el detalle caso por caso del modelo más chico.


In [23]:
from collections import Counter

def clasificar(texto, ejemplos=None, modelo=MODEL):
    # num_predict=10 porque pedimos UNA palabra. avisar=False: acá el corte es esperado
    # (el modelo querría seguir hablando) y no nos molesta, ya tenemos la etiqueta.
    return ask(build_prompt(texto, ejemplos), model=modelo, num_predict=10, avisar=False)

def norm_label(s):
    s = s.lower()
    for lab in ["positivo", "negativo", "neutral"]:
        if lab in s:
            return lab
    return "?"

CONDICIONES = [("zero", None), ("one", SHOTS_OK[:1]),
               ("few-OK", SHOTS_OK), ("few-RND", SHOTS_RANDOM)]
golds_sent = [g for _, g in EVAL_SENT]

def medir_shots(modelo):
    preds = {n: [norm_label(clasificar(t, ejs, modelo)) for t, _ in EVAL_SENT]
             for n, ejs in CONDICIONES}
    accs = {n: round(sum(p == g for p, g in zip(P, golds_sent)) / len(golds_sent), 2)
            for n, P in preds.items()}
    return preds, accs

medidas = {}
for nombre, modelo in ESCALAS3:
    print("midiendo", nombre, "(" + modelo + ") ...", flush=True)
    medidas[nombre] = medir_shots(modelo)

tabla("Accuracy por escala (n = " + str(len(EVAL_SENT)) + " casos)",
      [n for n, _ in CONDICIONES],
      [(nombre, medidas[nombre][1]) for nombre, _ in ESCALAS3])

# La accuracy sola no dice QUÉ salió mal. Esto sí: ¿el modelo usa las tres etiquetas,
# o se derrumba sobre una sola? Un modelo que contesta siempre lo mismo no está
# clasificando — está copiando un patrón del contexto.
print("Distribución de las predicciones (¿usa las 3 etiquetas o colapsa a una?)")
print("─" * 74)
for nombre, _ in ESCALAS3:
    preds = medidas[nombre][0]
    for cond, _ in CONDICIONES:
        cuentas = Counter(preds[cond]).most_common()
        txt = "  ".join(k + "×" + str(v) for k, v in cuentas)
        flag = "   <-- COLAPSO a una sola etiqueta" if len(cuentas) == 1 else ""
        print((nombre + " / " + cond).ljust(22), txt.ljust(34) + flag)
    print()

# Caso por caso, para la escala donde hay algo que mirar.
FOCO = "diminuto"
print("Predicciones de '" + FOCO + "' caso por caso:")
p = medidas[FOCO][0]
print("texto".ljust(38), "gold".ljust(9), "zero".ljust(9), "few-OK".ljust(9), "few-RND")
for i, (t, g) in enumerate(EVAL_SENT):
    print(t[:36].ljust(38), g.ljust(9), p["zero"][i].ljust(9),
          p["few-OK"][i].ljust(9), p["few-RND"][i])

RESULTADOS["few_shot"] = {nombre: medidas[nombre][1] for nombre, _ in ESCALAS3}

midiendo diminuto (qwen2.5:0.5b-instruct) ...
midiendo chico (qwen2.5:1.5b-instruct) ...
midiendo 4B (qwen3:4b-instruct) ...

Accuracy por escala (n = 12 casos)
────────────────────────────────────────────────────────
                   zero        one     few-OK    few-RND
diminuto           0.67       0.33       0.33       0.67
chico              0.83       0.92       0.92        1.0
4B                 0.92       0.92        1.0        1.0

Distribución de las predicciones (¿usa las 3 etiquetas o colapsa a una?)
──────────────────────────────────────────────────────────────────────────
diminuto / zero        positivo×7  negativo×5            
diminuto / one         negativo×12                          <-- COLAPSO a una sola etiqueta
diminuto / few-OK      negativo×12                          <-- COLAPSO a una sola etiqueta
diminuto / few-RND     negativo×6  positivo×4  neutral×2 

chico / zero           positivo×6  negativo×4  neutral×2 
chico / one            positivo×5  negativo×4 

**Cómo leerlo: tres regímenes, tres lecciones.**

*Arriba (4B): el techo.* Zero-shot ya resuelve la tarea, así que las cuatro condiciones se amontonan arriba y las diferencias son de un caso. Esa fila es la lección incómoda — *la técnica que te enseñan como imprescindible puede no mover nada si el modelo ya sabe hacer la tarea.*

*En el medio: el régimen de Min et al.* Es donde la técnica tiene de dónde agarrarse: zero-shot flojo, few-shot mejor. Y si `few-RND` —labels **barajados**— se parece a `few-OK`, viste el corazón del paper: las demostraciones fijan el **formato** y el espacio de labels, no el mapeo input→respuesta. Primera aparición del backbone **forma vs contenido** (el del tag `<think>`, Clase 6).

*Abajo: el colapso.* Mirá la tabla de distribuciones. Si a alguna escala le aparece un `COLAPSO a una sola etiqueta`, ahí tenés el modo de falla que nadie menciona: **por debajo de cierta capacidad, los ejemplos no ayudan, hacen daño**. El modelo no aprende la tarea desde las demostraciones — copia el patrón más superficial que encuentra (una etiqueta que vio y repite). Puede terminar por debajo de su propio zero-shot, y hasta puede pasar que los labels *barajados* le vayan mejor que los correctos, que es la clase de resultado que no tiene sentido hasta que entendés que no está clasificando en absoluto.

> Los ejemplos enseñan la forma de la tarea, no el contenido. Eso es lo bueno de few-shot: el modelo ve el molde Texto: … / Sentimiento: <una palabra> y lo copia. Pero copiar el molde y resolver la tarea compiten por la misma capacidad de generación. En un modelo grande, copiar el formato es trivial y le sobra todo para lo demás. En un modelo de 0.5B, copiar el formato le consume todo: el patrón superficial domina la distribución del próximo token y aplasta la señal semántica, que ya era débil.

> Dicho de otro modo: a esa escala el modelo no infiere "la tarea es clasificar sentimiento". Infiere "después de Sentimiento: va esta palabra que ya vi".

> La prueba más limpia la tenés en la columna one. El único ejemplo que le pasamos está etiquetado negativo — y el modelo contesta negativo doce veces seguidas. Un ejemplo, un label, doce copias. Eso no es una hipótesis sobre el mecanismo, es el mecanismo a la vista.

> En few-OK los tres ejemplos son negativo, positivo, neutral, y colapsa al primero. Eso apunta a sesgo de primacía, aunque con una sola observación lo diría como lectura plausible, no como hecho.

Por qué los labels barajados

Esas tres filas juntas dicen algo que ninguna dice sola: **few-shot no es magia ni es inútil — es una herramienta con un rango de operación**. Abajo del rango envenena, arriba del rango sobra.

> **Sobre el ruido.** Son 12 casos. Una diferencia de un caso son 0.08 de accuracy: eso es ruido, no señal. Fijate en diferencias grandes, y desconfiá de las chicas. Es el mismo criterio que vas a necesitar para leer cualquier benchmark.

▶ **Láminas 4–5** (el espectro; qué enseñan los ejemplos). Puntero: **lámina 6** (el in-context learning emerge del pretraining).


## 4 · Claro vs vago — y el juez, bien usado

Para una tarea **abierta** (sin respuesta única verificable) volvemos al harness con LLM-as-judge. Comparamos un prompt vago contra uno claro y específico.

Pero esta sección hace dos cosas a la vez, y la segunda es la más importante. Además del **score del juez** (blando, subjetivo) medimos un **verificador duro**: el prompt claro pide *exactamente 3 oraciones*, así que las contamos. Dos instrumentos sobre el mismo par de salidas — y la pregunta interesante es si coinciden.

> **Un detalle del diseño que importa.** La rúbrica que recibe el juez incluye el formato pedido ("en exactamente 3 oraciones"). No es un detalle: un juez al que le das un criterio objetivo se comporta muy distinto de uno al que le pedís "evaluá la calidad". Volvemos sobre esto abajo.


In [24]:
import re as _re

# 4 casos y no 2: con 2, un caso malo mueve el promedio 5 puntos.
dataset_expl = [{"tema": "por qué el cielo es azul"},
                {"tema": "qué es una tasa de interés"},
                {"tema": "por qué flotan los barcos"},
                {"tema": "qué es una vacuna"}]

def prompt_vago(tema):
    return "Contame sobre " + tema + "."

def prompt_claro(tema):
    return ("Sos un buen docente. Explicá en exactamente 3 oraciones, con una analogía "
            "cotidiana y sin jerga: " + tema + ".")

# num_predict amplio: en la versión anterior la respuesta vaga se cortaba y el juez la
# penalizaba POR ESTAR CORTADA. Eso no mide el prompt, mide el presupuesto de tokens.
def run_vago(inp):  return ask(prompt_vago(inp["tema"]),  num_predict=1500)
def run_claro(inp): return ask(prompt_claro(inp["tema"]), num_predict=1500)

def n_oraciones(t):
    return len([s for s in _re.split(r"[.!?]+", t) if len(s.strip()) > 10])

crit = "Claridad, correctitud, presencia de una analogía, y respeto del formato pedido."
tarea_expl = "Explicar un concepto con claridad, en exactamente 3 oraciones."
r_vago  = evaluator.run_evaluation(run_vago,  dataset_expl,
                                   task_description=tarea_expl, extra_criteria=crit)
r_claro = evaluator.run_evaluation(run_claro, dataset_expl,
                                   task_description=tarea_expl, extra_criteria=crit)

# Ejemplo concreto (primer caso).
tema0 = dataset_expl[0]["tema"]
mostrar_ejemplo("VAGO — prompt + respuesta", prompt_vago(tema0), r_vago[0]["output"])
print("   juez:", r_vago[0]["score"], "/10 —", r_vago[0]["reasoning"][:200], "...\n")
mostrar_ejemplo("CLARO — prompt + respuesta", prompt_claro(tema0), r_claro[0]["output"])
print("   juez:", r_claro[0]["score"], "/10 —", r_claro[0]["reasoning"][:200], "...\n")

# Caso por caso: el promedio solo esconde de dónde sale. Y al lado del juez (blando),
# un verificador DURO del formato pedido: ¿son 3 oraciones o no?
print("caso".ljust(30), "juez vago", "  ", "juez claro", "  ", "orac. vago", "  ", "orac. claro")
for i, c in enumerate(dataset_expl):
    print(c["tema"][:28].ljust(30),
          str(r_vago[i]["score"]).rjust(9), "  ", str(r_claro[i]["score"]).rjust(10), "  ",
          str(n_oraciones(r_vago[i]["output"])).rjust(10), "  ",
          str(n_oraciones(r_claro[i]["output"])).rjust(11))

juez_vago  = round(sum(r["score"] for r in r_vago) / len(r_vago), 2)
juez_claro = round(sum(r["score"] for r in r_claro) / len(r_claro), 2)
fmt_vago   = sum(1 for r in r_vago  if n_oraciones(r["output"]) == 3)
fmt_claro  = sum(1 for r in r_claro if n_oraciones(r["output"]) == 3)
largo_vago  = round(sum(len(r["output"]) for r in r_vago) / len(r_vago))
largo_claro = round(sum(len(r["output"]) for r in r_claro) / len(r_claro))

tabla("Vago vs claro",
      ["score juez", "3 oraciones", "largo medio"],
      [("vago",  {"score juez": juez_vago,  "3 oraciones": str(fmt_vago) + "/4",  "largo medio": largo_vago}),
       ("claro", {"score juez": juez_claro, "3 oraciones": str(fmt_claro) + "/4", "largo medio": largo_claro})])

RESULTADOS["claro_vs_vago"] = {"juez vago": juez_vago, "juez claro": juez_claro,
                               "formato vago": str(fmt_vago) + "/4",
                               "formato claro": str(fmt_claro) + "/4"}

──────────────────────────────────────────────────────────────────────
VAGO — prompt + respuesta
──────────────────────────────────────────────────────────────────────
PROMPT:
Contame sobre por qué el cielo es azul.

RESPUESTA DEL MODELO:
¡Claro! La pregunta "¿Por qué el cielo es azul?" es una de las más famosas y hermosas en la ciencia. Aunque parece simple, tiene una explicación fascinante que combina física y química. Aquí te lo explico de forma clara y sencilla:

---

### 🌤️ ¿Por qué el cielo es azul?

El cielo parece azul porque **la luz del sol se dispersa** (se rompe en pequeñas partes) cuando pasa por la atmósfera de la Tierra, y **el azul se dispersa más que otros colores**.

---

### 🔍 ¿Cómo funciona esto?

1. **La luz del sol no es de un solo color**  
   El sol emite luz que contiene todos los colores del arcoíris: rojo, naranja, amarillo, verde, azul, violeta, etc. Esta luz se llama **luz blanca**.

2. **La atmósfera está hecha de moléculas**  
   La atmósfera está compues

**Lo que acaba de pasar, en dos niveles.**

*El nivel de la técnica.* La columna **"3 oraciones"** no admite interpretación: el prompt claro pide un formato y o lo cumple o no. Ahí, "instrucciones claras y específicas" deja de ser un consejo de buena onda y se vuelve verificable. Y mirá el **largo medio**: el prompt vago produce respuestas varias veces más largas. Más texto no es más calidad — es menos control.

*El nivel del instrumento.* Comparás ahora las dos columnas de medición. Si el juez y el verificador duro **coinciden**, no es casualidad: es porque la rúbrica incluía un criterio objetivo ("exactamente 3 oraciones") que el juez pudo aplicar casi mecánicamente. Esa es la moraleja práctica y es accionable: **un LLM-as-judge es mucho mejor evaluador cuando le das criterios verificables que cuando le pedís que opine sobre "calidad".** La rúbrica hace más por la confiabilidad del juez que el tamaño del modelo.

Y guardá la contracara, porque la vas a ver en la Sección 16: cuando el criterio objetivo es **aritmético** —contar caracteres— el juez lo lee, dice que lo aplica, y no puede. Ahí el reparto deja de ser una buena práctica y pasa a ser obligatorio.

> **Si en tu corrida el juez y el verificador NO coinciden** —por ejemplo si premió a la respuesta larga que incumple el formato— no lo tapes: acabás de ver el **sesgo de verbosidad**, uno de los sesgos documentados del LLM-as-judge (Clase 6). Es el mismo mensaje por el camino contrario: el juez es un instrumento con sesgos, no un oráculo.

▶ **Lámina 7** (los dos principios del craft) y **Clase 6** (los límites del LLM-as-judge).


## 5 · Chain-of-Thought — y por qué más tokens es más cómputo

Sobre problemas con respuesta numérica (**verificable** → accuracy directa), comparamos pedir la respuesta directa vs pedir el **paso a paso** (Chain-of-Thought). Y de paso medimos el **largo** de la salida, para ver que el razonamiento *es* cómputo: más tokens generados = más forward passes donde apoyar el cálculo.


In [25]:
import re

PROBLEMAS = [
    ("Un colectivo sale con 12 pasajeros. En la primera parada suben 9 y bajan 4. "
     "En la segunda suben 3 y bajan 7. ¿Cuántos pasajeros quedan?", 13),
    ("Ana tiene 3 cajas con 8 lápices cada una y regala 5 lápices. ¿Cuántos le quedan?", 19),
    ("Un libro tiene 240 páginas. Leí 1/4 el lunes y 1/3 del resto el martes. "
     "¿Cuántas páginas leí en total?", 120),
    ("Compré 5 alfajores a $350 cada uno y pagué con $2000. ¿Cuánto vuelto recibí?", 250),
    ("Hay 30 alumnos. 2/5 juegan al fútbol y la mitad del resto, al básquet. "
     "¿Cuántos juegan al básquet?", 9),
    ("Un tanque tiene 100 litros. Se llena a 8 L/min y se vacía a 3 L/min. "
     "¿Cuántos minutos para llenarlo desde vacío?", 20),
]

def extract_int(s):
    if re.search(r"[Rr]espuesta\s*:", s):
        s = re.split(r"[Rr]espuesta\s*:", s)[-1]
    # Sin el signo: con "-?" un guión de lista o de resta se colaba como número negativo
    # y ensuciaba el voto de la Sección 7 (aparecía un -1 fantasma).
    nums = re.findall(r"\d+", s)
    return int(nums[-1]) if nums else None

def sin_marcador(outs):
    # Salidas que nunca llegaron a "Respuesta:". Si una cadena se trunca, extract_int cae
    # al fallback "último número del texto" y devuelve un valor INTERMEDIO del cálculo:
    # se contaría como error de razonamiento cuando en realidad faltó presupuesto.
    return sum(1 for o in outs if not re.search(r"[Rr]espuesta\s*:", o))

# Estos dos prompts se usan ACÁ y también en la Sección 8: definirlos una sola vez es lo
# que hace comparables los números de las dos secciones.
#
# OJO con p_directo: tiene que ser DIRECTO de verdad. Con un "Respondé sólo con el número"
# a secas, el modelo igual escribe "12 - 4 + 9 - 7 + 3 = 13" — o sea, hace el cálculo en
# la salida. Eso ya es Chain-of-Thought encubierto, y arruina el contraste de esta sección
# (mediríamos CoT contra CoT). Hay que prohibir explícitamente el cálculo, y además topear
# num_predict para que no tenga lugar donde hacerlo.
def p_directo(q):
    return (q + "\nRespondé ÚNICAMENTE con el número final. "
                "Sin cálculos, sin pasos, sin explicación, sin unidades.")

def p_cot(q):
    return q + "\nPensá paso a paso y al final escribí 'Respuesta: <número>'."

def correr5(modo, modelo=MODEL):
    build  = p_directo if modo == "directo" else p_cot
    np_tok = NP_DIR if modo == "directo" else NP_COT
    outs  = [ask(build(q), num_predict=np_tok, model=modelo,
                 avisar=(modo != "directo")) for q, _ in PROBLEMAS]
    preds = [extract_int(o) for o in outs]
    golds = [a for _, a in PROBLEMAS]
    acc   = sum(p == g for p, g in zip(preds, golds)) / len(golds)
    largo = sum(len(o.split()) for o in outs) / len(outs)   # palabras ~ proxy de tokens
    return outs, round(acc, 2), round(largo, 1)

outs_dir, acc_dir, len_dir = correr5("directo")
outs_cot, acc_cot, len_cot = correr5("cot")

q0 = PROBLEMAS[0][0]
mostrar_ejemplo("DIRECTO — prompt + respuesta", p_directo(q0), outs_dir[0])
mostrar_ejemplo("CoT — prompt + respuesta",     p_cot(q0),     outs_cot[0])

tabla("Directo vs Chain-of-Thought (n = " + str(len(PROBLEMAS)) + ")",
      ["accuracy", "palabras/resp"],
      [("directo", {"accuracy": acc_dir, "palabras/resp": len_dir}),
       ("CoT",     {"accuracy": acc_cot, "palabras/resp": len_cot})])
print("cadenas de CoT sin 'Respuesta:':", sin_marcador(outs_cot), "de", len(outs_cot))

RESULTADOS["cot"] = {"directo": acc_dir, "CoT": acc_cot,
                     "palabras directo": len_dir, "palabras CoT": len_cot}

──────────────────────────────────────────────────────────────────────
DIRECTO — prompt + respuesta
──────────────────────────────────────────────────────────────────────
PROMPT:
Un colectivo sale con 12 pasajeros. En la primera parada suben 9 y bajan 4. En la segunda suben 3 y bajan 7. ¿Cuántos pasajeros quedan?
Respondé ÚNICAMENTE con el número final. Sin cálculos, sin pasos, sin explicación, sin unidades.

RESPUESTA DEL MODELO:
10

──────────────────────────────────────────────────────────────────────
CoT — prompt + respuesta
──────────────────────────────────────────────────────────────────────
PROMPT:
Un colectivo sale con 12 pasajeros. En la primera parada suben 9 y bajan 4. En la segunda suben 3 y bajan 7. ¿Cuántos pasajeros quedan?
Pensá paso a paso y al final escribí 'Respuesta: <número>'.

RESPUESTA DEL MODELO:
Vamos paso a paso:

1. **Inicialmente**, el colectivo sale con **12 pasajeros**.

2. **En la primera parada**:
   - Suben **9** pasajeros.
   - Bajan **4** pasajeros.


**El Aha!, medido.** CoT no le agrega conocimiento al modelo: le da **lugar para computar** antes de comprometerse. Fijate que la salida de CoT es mucho más larga — esos tokens intermedios son el cálculo hecho visible. Es la misma física de la Clase 6: más tokens = más cómputo.

> **Una objeción honesta, porque alguien la va a hacer.** "Al prompt directo le prohibiste calcular: lo lisiaste para que CoT ganara." Es cierto que la restricción es deliberada, y conviene entender por qué es necesaria. Con un "respondé sólo el número" a secas, el modelo igual escribe `12 - 4 + 9 - 7 + 3 = 13`: hace el cálculo en la salida. Eso **ya es** Chain-of-Thought, sólo que sin pedirlo — y entonces estaríamos comparando CoT contra CoT y midiendo cero diferencia 

> Prohibir los pasos no es hacer trampa: es lo que permite aislar la variable. La pregunta de la sección no es "¿el modelo sabe resolver esto?" sino "¿cuánto de su desempeño depende de tener tokens donde apoyar el cálculo?". Para responderla hay que sacarle esos tokens. Y el resultado —que sin lugar para computar se derrumba— es exactamente la tesis: el razonamiento no está sólo en los pesos, está en el **espacio de generación** que le dejes.

> Sin tokens intermedios, el modelo tiene que resolver cuatro operaciones encadenadas dentro de la profundidad fija de un solo forward pass. No tiene dónde apoyar los parciales: no existe un scratchpad interno, lo único que puede usar como memoria de trabajo es lo que ya escribió — y le prohibimos escribir. 
Mirá el error, que es elocuente: 12+9−4−7 = 10. Parece que aplicó tres de las cuatro operaciones y perdió el +3 en el camino. No es que no sepa sumar; es que no pudo sostener el estado intermedio.

▶ **Láminas 10–12** (la grilla shots×razonamiento; Chain-of-Thought; tokens como cómputo).


## 6 · Zero-shot CoT y RE2 (releer la pregunta)

Dos técnicas más, sobre los mismos problemas. **Zero-shot CoT**: la frase gatillo "pensemos paso a paso", sin ejemplos. **RE2** (re-reading): repetir la pregunta antes de responder. RE2 gasta cómputo en el **input** —y el porqué es lindo: un decoder-only es causal, así que releer deja que la segunda pasada atienda a la pregunta entera (comprensión "bidireccional" desde el prompt).


In [26]:
def build_p6(pregunta, modo):
    if modo == "zscot":
        return pregunta + "\nPensemos paso a paso. Al final escribí 'Respuesta: <número>'."
    return ("Pregunta: " + pregunta + "\nLeé la pregunta de nuevo: " + pregunta +
            "\nPensemos paso a paso. Al final escribí 'Respuesta: <número>'.")

def correr6(modo):
    outs  = [ask(build_p6(q, modo), num_predict=NP_COT) for q, _ in PROBLEMAS]
    preds = [extract_int(o) for o in outs]
    golds = [a for _, a in PROBLEMAS]
    return outs, round(sum(p == g for p, g in zip(preds, golds)) / len(golds), 2)

outs_zs,  acc_zs  = correr6("zscot")
outs_re2, acc_re2 = correr6("re2")

q0 = PROBLEMAS[0][0]
mostrar_ejemplo("ZERO-SHOT CoT — prompt + respuesta", build_p6(q0, "zscot"), outs_zs[0])
mostrar_ejemplo("RE2 (releer la pregunta) — prompt + respuesta", build_p6(q0, "re2"), outs_re2[0])

tabla("Gatillo de razonamiento vs releer la pregunta (n = " + str(len(PROBLEMAS)) + ")",
      ["accuracy", "sin 'Respuesta:'"],
      [("zero-shot CoT", {"accuracy": acc_zs,  "sin 'Respuesta:'": sin_marcador(outs_zs)}),
       ("RE2 + CoT",     {"accuracy": acc_re2, "sin 'Respuesta:'": sin_marcador(outs_re2)})])

RESULTADOS["zscot_re2"] = {"zero-shot CoT": acc_zs, "RE2+CoT": acc_re2}

──────────────────────────────────────────────────────────────────────
ZERO-SHOT CoT — prompt + respuesta
──────────────────────────────────────────────────────────────────────
PROMPT:
Un colectivo sale con 12 pasajeros. En la primera parada suben 9 y bajan 4. En la segunda suben 3 y bajan 7. ¿Cuántos pasajeros quedan?
Pensemos paso a paso. Al final escribí 'Respuesta: <número>'.

RESPUESTA DEL MODELO:
Vamos a resolver el problema paso a paso.

**Iniciamos con:**  
El colectivo sale con **12 pasajeros**.

---

**Primera parada:**  
- Suben **9** pasajeros.  
  Entonces, pasajeros ahora = 12 + 9 = **21**  
- Bajan **4** pasajeros.  
  Entonces, pasajeros ahora = 21 - 4 = **17**

---

**Segunda parada:**  
- Suben **3** pasajeros.  
  Entonces, pasajeros ahora = 17 +  3 = **20**  
- Bajan **7** pasajeros.  
  Entonces, pasajeros ahora = 20 - 7 = **13**

---

**Resultado final:**  
Los pasajeros que quedan en el colectivo son **13**.

Respuesta: 13

───────────────────────────────────────

**Los detalles finos, y por qué RE2 no es simplemente "más tokens".**

Un decoder-only es **causal** (Clase 2): el token en la posición *i* sólo puede atender a las posiciones ≤ *i*, nunca hacia adelante. No es un defecto — es lo que vuelve entrenable la predicción del próximo token. Pero tiene una consecuencia concreta: cuando el modelo lee la pregunta **por primera vez**, cada token se codifica sin saber lo que viene después. En `Hay 30 alumnos. 2/5 juegan al fútbol y la mitad del resto, al básquet. ¿Cuántos juegan al básquet?`, el fragmento "Hay 30 alumnos" se procesa sin saber todavía que la pregunta va a ser sobre básquet, ni que "el resto" va a importar.

**Qué cambia al releer.** En la segunda copia, cada token de la pregunta puede atender a la **primera copia entera** — incluida la parte que en el original venía después. El "Hay 30 alumnos" de la segunda lectura se codifica ya sabiendo adónde va la pregunta. Recuperás una comprensión efectivamente **bidireccional** (estilo BERT) dentro de un modelo unidireccional, sin tocar el modelo: sólo con el prompt. No le agregaste información —la pregunta es idéntica— le agregaste la posibilidad de mirarla entera antes de empezar a responder.

**El control que confirma el mecanismo.** Si RE2 funcionara sólo por alargar el input, repetir *cualquier cosa* ayudaría. No pasa: repetir la **pregunta** ayuda, y repetir el gatillo de razonamiento ("pensemos paso a paso") dos veces, **no**. La ganancia está atada específicamente a re-codificar el enunciado. Es el tipo de control que separa una explicación de una historia plausible.

**Dónde gasta el cómputo.** Chain-of-Thought y RE2 son la misma palanca —más tokens = más cómputo— aplicada en puntas opuestas de la secuencia:

| | Gasta en | Cómo se ejecuta |
|---|---|---|
| Chain-of-Thought | la **salida** (decode) | serial: cada token espera al anterior |
| RE2 | el **input** (prefill) | paralelo: todas las posiciones a la vez |

Por eso RE2 es de las técnicas más baratas en tiempo real que existen. Y por eso **compone** con CoT en vez de competir: atacan cosas distintas —entender la pregunta vs. pensar la respuesta— y se usan juntas, que es exactamente lo que hace la celda de arriba.

> **El límite.** Duplicar una pregunta de 40 tokens sale gratis. Duplicar un documento de 5.000, no: ahí RE2 choca de frente con el presupuesto finito de atención de la Sección 15. Rinde cuando lo que releés es corto y denso.

> **Sobre estos números.** Son 6 problemas: una diferencia de 0.17 es **un caso**. Eso es ruido, no señal — y no se arregla con `SMOKE_TEST=False`, que sube el presupuesto de tokens y no la cantidad de problemas. Para medir el efecto de verdad hacen falta decenas de casos; el paper lo hace sobre varios benchmarks y modelos, y la ganancia real es modesta. Acá la demo sirve para ver el **artefacto** —un prompt con la pregunta repetida, que se entiende de una— y para entender el mecanismo.

▶ **Láminas 13–14** (releer la pregunta / cómputo en el input; zero-shot CoT).


## 7 · Self-consistency: muestrear y votar

Una cadena de razonamiento es una **muestra**, no una verdad. Si generamos varias (con `temperature > 0`) y nos quedamos con la respuesta **mayoritaria**, filtramos el ruido de un camino aislado. Comparamos una sola muestra vs el voto de N.


In [27]:
from collections import Counter

def build_p7(pregunta):
    return pregunta + "\nPensá paso a paso. Al final escribí 'Respuesta: <número>'."

def una_cadena(pregunta, modelo):
    return ask(build_p7(pregunta), temperature=0.8, num_predict=NP_COT, model=modelo)

def voto(pregunta, modelo, n=None):
    """Devuelve (nums, ganador, hubo_mayoria). Si ninguna respuesta se repite NO hay
    mayoría: devolver igual 'la primera' sería inventar un ganador."""
    n = n or N_CHAINS
    outs = [una_cadena(pregunta, modelo) for _ in range(n)]
    nums = [x for x in (extract_int(o) for o in outs) if x is not None]
    if not nums:
        return nums, None, False
    (top, veces), = Counter(nums).most_common(1)
    return nums, top, veces > 1

golds_num = [a for _, a in PROBLEMAS]
q0, gold0 = PROBLEMAS[0]

# EL contraste de la sección: las N respuestas de cada escala sobre el MISMO problema.
# Si todas son iguales, votar no puede aportar nada. La dispersión es la materia prima.
print("Problema:", q0[:70], "...  (correcto:", str(gold0) + ")\n")
resumen = {}
for nombre, modelo in ESCALAS:
    nums, ganador, hubo = voto(q0, modelo)
    print(nombre.ljust(10), "cadenas ->", nums)
    print(" " * 10, "distintas:", len(set(nums)), "| voto:",
          (str(ganador) if hubo else "SIN MAYORÍA (ninguna respuesta se repitió)"))
    resumen[nombre] = nums

# Accuracy sobre todos los problemas: una sola muestra vs el voto, por escala.
filas = []
for nombre, modelo in ESCALAS:
    single = [extract_int(una_cadena(q, modelo)) for q, _ in PROBLEMAS]
    votos, sin_may = [], 0
    for q, _ in PROBLEMAS:
        _, g, hubo = voto(q, modelo)
        votos.append(g)
        sin_may += (not hubo)
    acc = lambda P: round(sum(p == g for p, g in zip(P, golds_num)) / len(golds_num), 2)
    filas.append((nombre, {"1 muestra": acc(single), "voto de " + str(N_CHAINS): acc(votos),
                           "sin mayoría": str(sin_may) + "/" + str(len(PROBLEMAS))}))

tabla("Self-consistency por escala (n = " + str(len(PROBLEMAS)) + " problemas)",
      ["1 muestra", "voto de " + str(N_CHAINS), "sin mayoría"], filas)

RESULTADOS["self_consistency"] = dict(filas)
RESULTADOS["self_consistency_cadenas"] = resumen

Problema: Un colectivo sale con 12 pasajeros. En la primera parada suben 9 y baj ...  (correcto: 13)

   [⚠ cortado por num_predict=1024]
diminuto   cadenas -> [14, 4, 12, 11, 2]
           distintas: 5 | voto: SIN MAYORÍA (ninguna respuesta se repitió)
4B         cadenas -> [13, 13, 13, 13, 13]
           distintas: 1 | voto: 13

Self-consistency por escala (n = 6 problemas)
──────────────────────────────────────────────────────
                 1 muestra     voto de 5   sin mayoría
diminuto               0.0          0.17           0/6
4B                    0.83           1.0           0/6



**La palanca general.** Gastar más cómputo en inferencia —muestrear y agregar— compra confiabilidad. Pero fijate en la condición de posibilidad: **el voto sólo puede ayudar si las cadenas discrepan.**

Mirá las dos listas de respuestas de arriba. Si el 4B devolvió el mismo número cinco veces, votar es gastar cinco veces más cómputo para llegar exactamente al mismo lado: sin dispersión no hay nada que agregar. En el modelo chico, donde las cadenas se van por caminos distintos, el voto tiene de dónde sacar la señal.

La lección no es "self-consistency funciona", es más fina: **es una técnica para convertir varianza en precisión.** Si no hay varianza, no hay nada que convertir. Igual que con few-shot en la Sección 3, la técnica vale en el régimen donde el modelo todavía falla.

▶ **Lámina 15** (self-consistency). Puntero: **lámina 16** (de la cadena al árbol: Tree-of-Thought).


## 8 · El giro: el CoT manual es el ancestro del razonamiento de RL

Todo lo anterior lo corrimos sobre el modelo *instruct*, que **no** razona: por eso el CoT manual lo ayudaba. Ahora comparamos **dos modelos de la misma familia** sobre los mismos problemas: el **instruct** (respuesta directa) contra el **modelo de razonamiento** (`MODEL_THINK`, con `think=True`), que piensa solo antes de responder — esa capacidad la obtuvo mediante el RL (Clase 6).

Como bonus, con `return_thinking=True` podemos **ver el monólogo** que el modelo genera por su cuenta: es el CoT manual de las secciones anteriores, ahora internalizado.


In [28]:
# Reutilizamos p_directo / p_cot de la Sección 5: MISMO prompt, así los números de las
# dos secciones se pueden comparar.
def p_razonador(q): return q + "\nAl final escribí 'Respuesta: <número>'."

outs_s8_dir = [ask(p_directo(q), num_predict=NP_DIR, avisar=False) for q, _ in PROBLEMAS]
outs_s8_raz = [ask(p_razonador(q), model=MODEL_THINK, think=True, num_predict=2048)
               for q, _ in PROBLEMAS]

golds_num = [a for _, a in PROBLEMAS]

def acc_outs(outs):
    return round(sum(extract_int(o) == g for o, g in zip(outs, golds_num)) / len(golds_num), 2)

q0 = PROBLEMAS[0][0]
mostrar_ejemplo("INSTRUCT (directo) — prompt + respuesta", p_directo(q0), outs_s8_dir[0])
mostrar_ejemplo("RAZONADOR (think=True) — prompt + respuesta", p_razonador(q0), outs_s8_raz[0])

tabla("Instruct vs razonamiento nativo (n = " + str(len(PROBLEMAS)) + ")",
      ["accuracy"],
      [("instruct directo",     {"accuracy": acc_outs(outs_s8_dir)}),
       ("razonador think=True", {"accuracy": acc_outs(outs_s8_raz)})])

RESULTADOS["razonador"] = {"instruct directo": acc_outs(outs_s8_dir),
                           "think=True": acc_outs(outs_s8_raz)}

──────────────────────────────────────────────────────────────────────
INSTRUCT (directo) — prompt + respuesta
──────────────────────────────────────────────────────────────────────
PROMPT:
Un colectivo sale con 12 pasajeros. En la primera parada suben 9 y bajan 4. En la segunda suben 3 y bajan 7. ¿Cuántos pasajeros quedan?
Respondé ÚNICAMENTE con el número final. Sin cálculos, sin pasos, sin explicación, sin unidades.

RESPUESTA DEL MODELO:
10

──────────────────────────────────────────────────────────────────────
RAZONADOR (think=True) — prompt + respuesta
──────────────────────────────────────────────────────────────────────
PROMPT:
Un colectivo sale con 12 pasajeros. En la primera parada suben 9 y bajan 4. En la segunda suben 3 y bajan 7. ¿Cuántos pasajeros quedan?
Al final escribí 'Respuesta: <número>'.

RESPUESTA DEL MODELO:
Para resolver el problema, seguimos los pasos:

1. **Iniciamos con 12 pasajeros** en el colectivo.
2. **En la primera parada**:
   - Bajan 4 pasajeros: $12 -

In [29]:
# ¿Y qué PENSÓ el razonador? El .content de arriba salió limpio; el monólogo va en .thinking.
resp, pensamiento = ask(p_razonador(PROBLEMAS[0][0]), model=MODEL_THINK, think=True,
                        num_predict=2048, return_thinking=True)
print("PENSAMIENTO (primeros 600 chars):\n", pensamiento[:600], "\n")
print("RESPUESTA:", resp)

PENSAMIENTO (primeros 600 chars):
 Okay, let's see. So there's this problem about a bus group that starts with 12 passengers. Then at the first stop, 9 get on and 4 get off. Then at the second stop, 3 get on and 7 get off. We need to find out how many passengers are left at the end.

Hmm, let me break this down step by step. First, the initial number of passengers is 12. Then at the first stop, 9 people get on. Wait, but does that mean they were already on the bus? Wait, no, the problem says "un colectivo sale con 12 pasajeros" which translates to "a bus leaves with 12 passengers". So the bus starts with 12 people.

Then at the 

RESPUESTA: Para resolver el problema, seguimos los pasos:

1. **Iniciamos con 12 pasajeros** en el colectivo.
2. **En la primera parada**:
   - Bajan 4 pasajeros: $12 - 4 = 8$.
   - Suben 9 pasajeros: $8 + 9 = 17$.
3. **En la segunda parada**:
   - Bajan 7 pasajeros: $17 - 7 = 10$.
   - Suben 3 pasajeros: $10 + 3 = 13$.

**Respuesta: 13**


**La lectura.** El modelo de razonamiento, sin que le pidamos ningún paso a paso, iguala o supera al instruct — y en el monólogo se ve *por qué*: hace exactamente el tipo de descomposición que en las secciones 5–7 teníamos que pedir a mano. Ese es el giro del deck: el CoT manual es el **ancestro** de lo que el RL internalizó. La **capacidad** vino del RL; la técnica de prompt sólo la simulaba desde afuera. Por eso, sobre un reasoning model, agregar CoT manual aporta poco. Guardá "capacidad vs forma": vuelve en structured outputs.

▶ **Lámina 17** (el giro con reasoning models).


## 9 · El pivote: de expandir a restringir

Todo lo anterior **expandió** el espacio de tokens: más cadenas, más muestras, más cómputo. Ahora la operación **inversa** sobre la misma distribución: **contraer** el espacio, forzando que la salida tenga exactamente la forma que necesitamos. Mismo objeto (la distribución del próximo token), direcciones opuestas.

▶ **Lámina 18** (las dos palancas sobre la misma distribución).


## 10 · El JSON "a mano" se rompe → structured outputs

Pedir JSON en el prompt es una **súplica**, no una garantía: el modelo es un predictor estocástico de tokens y nada lo obliga a devolver lo que pediste. Vamos a medir cuánto vale esa súplica, y después a reemplazarla por una garantía dura con `format=` + un schema de Pydantic.

**Por qué corremos dos escalas.** Una garantía que sólo vale para el modelo grande no es una garantía. Si `format=` funciona de verdad, el modelo diminuto tiene que cumplir el schema igual que el 4B — y si no, acá se va a ver. Es el mismo criterio de las Secciones 3 y 7: la técnica se juzga donde el modelo todavía falla.

---

**Las tres columnas de la tabla, explicadas.** Sobre cada salida cruda nos hacemos tres preguntas que es fácil confundir en un solo "¿anduvo?". Cada una falla por un motivo distinto y se arregla de forma distinta.

**1 · `json.loads` — ¿parsea la respuesta entera?**

Es literalmente lo que hace tu código si confiás en el modelo: agarrás el string que devolvió y se lo pasás al parser. Si el modelo antepuso "Aquí tienes:" o envolvió el objeto en un bloque de markdown, esto explota.

**2 · `extract_json` — ¿la rescata un parser tolerante?**

`extract_json` es una función que vive en `promptlab.py` y que el harness viene usando **desde la Sección 2** para poder leer el veredicto del juez. No es magia: son tres intentos en escalera.

```python
def extract_json(text):
    text = re.sub(r"^```(json)?", "", text.strip()).strip()  # 1. saca el fence de apertura
    text = re.sub(r"```$", "", text).strip()                 #    y el de cierre
    try:
        return json.loads(text)                              # 2. prueba con el texto pelado
    except Exception:
        pass
    for abre, cierra in [("[", "]"), ("{", "}")]:            # 3. si no, recorta desde el primer
        i, j = text.find(abre), text.rfind(cierra)           #    '{' hasta el último '}' y prueba
        if i != -1 and j != -1 and j > i:
            try:
                return json.loads(text[i:j + 1])
            except Exception:
                continue
    return None                                              # 4. se rinde
```

Eso es el **código de rescate**: veinte líneas que escribís, mantenés y testeás, y que existen sólo porque el modelo no tiene ninguna obligación de darte JSON. Si esta columna dice "sí", el JSON estaba ahí — tapado, pero ahí.

**3 · `claves+tipos` — ¿el objeto rescatado cumple el contrato?**

Que algo parsee no quiere decir que sea *lo que pediste*. Esta columna chequea a mano lo que un schema chequearía solo:

```python
ok_schema = (isinstance(obj, dict)                          # ¿es un objeto, y no una lista o un número?
             and set(obj) == {"tema", "dificultad"}         # ¿exactamente esas dos claves, ni una más?
             and isinstance(obj.get("dificultad"), int)     # ¿'dificultad' es un entero de verdad
             and not isinstance(obj.get("dificultad"), bool))  #   y no el string "3" ni un booleano?
```

> El `not isinstance(..., bool)` no es paranoia: en Python `bool` **hereda** de `int`, así que `isinstance(True, int)` devuelve `True`. Sin esa línea, un `"dificultad": true` pasaría el chequeo sin chistar. Es el tipo de detalle que un schema de Pydantic te ahorra tener que recordar.


In [30]:
import json

# Pedimos JSON por prompt y medimos si se puede CONFIAR en la respuesta cruda.
# Dos escalas a propósito: una garantía que sólo vale para el modelo grande no es
# una garantía. Es el mismo criterio de las Secciones 3 y 7.
def pedido_limpio(tema):
    return "Dame un objeto JSON con las claves 'tema' y 'dificultad' (1-5) para: " + tema + "."

def pedido_con_prosa(tema):
    return pedido_limpio(tema) + " Además, explicá brevemente tu elección."

TEMAS_JSON = ["fotosíntesis", "derivadas", "la Revolución Francesa"]

def evaluar_crudo(raw):
    """TRES preguntas distintas sobre la misma salida. Confundirlas es el error:
    cada una falla por un motivo diferente y se arregla de forma diferente."""
    try:
        json.loads(raw)                       # (1) ¿parsea la respuesta ENTERA?
        directo = True
    except Exception:
        directo = False
    obj = promptlab.extract_json(raw)         # (2) ¿la rescata el parser tolerante?
    ok_schema = (isinstance(obj, dict)        # (3) ¿el objeto cumple lo que pedimos?
                 and set(obj) == {"tema", "dificultad"}
                 and isinstance(obj.get("dificultad"), int)
                 and not isinstance(obj.get("dificultad"), bool))
    return directo, obj is not None, ok_schema

CONDICIONES_JSON = [
    ("diminuto / limpio", MODEL_TINY, pedido_limpio),
    ("4B / limpio",       MODEL,      pedido_limpio),
    ("4B / + prosa",      MODEL,      pedido_con_prosa),
]

# Una salida cruda de cada escala, para ver el artefacto antes de cualquier promedio.
for etiqueta, modelo, arma in CONDICIONES_JSON[:2]:
    mostrar_ejemplo("JSON A MANO — " + etiqueta,
                    arma("fotosíntesis"),
                    ask(arma("fotosíntesis"), model=modelo, temperature=0.7, num_predict=600))

filas_json, res_json = [], {}
print("Detalle por caso (x = falla):")
print("condición".ljust(20), "tema".ljust(24), "json.loads".rjust(11),
      "extract_json".rjust(13), "claves+tipos".rjust(13))
for etiqueta, modelo, arma in CONDICIONES_JSON:
    d = r = s = 0
    for tema in TEMAS_JSON:
        raw = ask(arma(tema), model=modelo, temperature=0.7, num_predict=600)
        a, b, c = evaluar_crudo(raw)
        d, r, s = d + a, r + b, s + c
        print(etiqueta.ljust(20), tema[:22].ljust(24),
              ("sí" if a else "x").rjust(11), ("sí" if b else "x").rjust(13),
              ("sí" if c else "x").rjust(13))
    n = len(TEMAS_JSON)
    fila = {"json.loads": "%d/%d" % (d, n), "extract_json": "%d/%d" % (r, n),
            "claves+tipos": "%d/%d" % (s, n)}
    filas_json.append((etiqueta, fila))
    res_json[etiqueta] = fila

tabla("Pedir JSON por prompt: ¿se puede confiar en la salida cruda?",
      ["json.loads", "extract_json", "claves+tipos"], filas_json)

RESULTADOS["json_a_mano"] = res_json

──────────────────────────────────────────────────────────────────────
JSON A MANO — diminuto / limpio
──────────────────────────────────────────────────────────────────────
PROMPT:
Dame un objeto JSON con las claves 'tema' y 'dificultad' (1-5) para: fotosíntesis.

RESPUESTA DEL MODELO:
Aquí tienes un objeto JSON con las claves `tema` y `dificultad` (1-5) para la tarea de fotosíntesis:

```json
{
  "tema": 3,
  "dificultad": 5
}
```

Aquí es cómo se representaría el objeto JSON en un idioma claro:

- `tema`: Este es el nombre del tema o tema a la que se le hará la fotosíntesis.
- `dificultad`: Este se refiere a la dificultad o grado de la tarea. En este caso, se indica que se hará la fotosíntesis con un nivel de dificultad de 5, lo que significa que el proceso de fotosíntesis será un trabajo muy complejo y exigente.

──────────────────────────────────────────────────────────────────────
JSON A MANO — 4B / limpio
──────────────────────────────────────────────────────────────────────
PRO

**Cómo leer la tabla: cada fila es un experimento distinto.**

**`4B / limpio` — el control honesto.** Un pedido simple, sin pedirle prosa. Si esta fila da 3/3 en `json.loads`, el modelo bueno te devuelve JSON pelado y directo, sin envoltorio.

Y eso hay que decirlo, porque cambia el argumento de la sección: **la tesis no es "siempre falla"**. Es más modesta y mucho más difícil de refutar: *no tenés ninguna garantía de que no falle*. Un pipeline que hoy anda con este prompt, este modelo y esta temperatura puede romperse mañana con otro prompt, otra versión del modelo o un caso raro — y no hay forma de saberlo hasta que pase en producción.

**`4B / + prosa` — el envoltorio, provocado a propósito.** Acá le pedimos que *además* explique su elección, así que el JSON viene envuelto sí o sí. No es una falla del modelo: es exactamente lo que le pedimos, y está para ver el problema aislado. Fijate que `extract_json` y `claves+tipos` siguen en verde — el objeto está perfecto, lo único roto es que no viene solo.

**`diminuto / limpio` — el fracaso de verdad.** Mismo pedido simple que la primera fila, otro modelo. Acá aparece lo que el envoltorio tapaba: si `extract_json` rescata pero `claves+tipos` se cae, el modelo produjo *un* JSON que no es **tu** JSON — un campo que debía ser string y vino como objeto anidado, un entero que vino como texto, claves de más.

Esa brecha entre las dos últimas columnas **es la diferencia entre "es JSON válido" y "es válido contra mi schema"** — o sea, entre *JSON mode* y *structured outputs*, los dos niveles de garantía que distingue la lámina 20. Verla medida vale más que leerla enunciada.

> **Lo que esta celda NO demuestra.** Que el modelo emita JSON sintácticamente roto —una comilla sin cerrar, una llave de más— pasa en producción, pero es **esporádico**: puede que en esta corrida no aparezca ni una vez. Eso no debilita el argumento, lo refuerza. Un fallo esporádico y sin cota es peor que uno sistemático: no lo reproducís, no lo testeás, y aparece el día que tenés mil llamadas por minuto.


### Imponer la forma en el decoder

**De la súplica a la garantía.** Lo de arriba midió cuánto vale *pedir* JSON por prompt: depende del modelo, del prompt y de la suerte. Ahora hacemos la operación inversa — en vez de pedir la forma, la **imponemos en el decoder** con `format=`. Y para describir esa forma vamos a usar **Pydantic**.

> **Qué es Pydantic — y por qué no tiene nada que ver con los LLMs.**
>
> Pydantic es una librería de **validación de datos** de Python, creada alrededor de 2017 por Samuel Colvin, o sea años antes de que existiera nada de esto. Su trabajo es mundano: declarás la forma de un dato con *type hints* normales de Python y la librería valida, convierte y serializa por vos. Es el motor de validación de **FastAPI**, se usa para leer archivos de configuración, para limpiar datos en pipelines de ETL y para tipar APIs. Desde la v2 (2023) su núcleo está escrito en Rust. Es una de las librerías más descargadas del ecosistema Python, y la enorme mayoría de quienes la usan nunca tocó un modelo de lenguaje.
>
> Aparece en esta clase por una coincidencia afortunada: **`model_json_schema()` emite un JSON Schema estándar** — el de `json-schema.org`, no un invento de Pydantic. Y JSON Schema es justamente el formato que Ollama, OpenAI y el resto esperan para describir la gramática a la que van a restringir el decoder. Pydantic es el **puente**: escribís una clase de Python y sale la especificación que el motor de inferencia necesita. Nada en Pydantic fue diseñado para esto; simplemente hablaba el idioma correcto cuando hizo falta.

**Los dos trabajos que hace la misma clase — y conviene no confundirlos.** `Ficha` va a aparecer dos veces en la celda de abajo, en dos momentos opuestos del ciclo:

| Método | Cuándo corre | Qué hace |
|---|---|---|
| `Ficha.model_json_schema()` | **antes** de generar | describe lo permitido; viaja en `format=` hasta el decoder |
| `Ficha.model_validate_json(raw)` | **después** de generar | valida el texto recibido y lo convierte en un objeto Python tipado |

Sin `format=`, Pydantic sólo puede hacer el segundo: revisar el desastre una vez que ocurrió. Ése es exactamente el modelo mental de *"el schema es un validador externo"* que la lámina 27 marca como equivocado. Con `format=`, la **misma clase** además restringe la generación: deja de ser un control de calidad al final de la línea y pasa a ser parte de cómo se produce la pieza.

**Un límite que conviene ver venir.** Lo que podés expresar en el schema está acotado por lo que Pydantic sabe describir: tipos (`str`, `int`, `list[int]`), conjuntos cerrados de valores (`Literal[...]`, que vamos a usar en la Sección 13) y cotas (`Field(max_length=60)`, que usamos acá abajo para que el campo `tema` no se desborde). Eso alcanza para muchísimo — pero no para todo. *"Que la dificultad sea coherente con el tema"* no es expresable en ningún schema.

**La forma se restringe; el sentido, no.** Miralo en la salida de abajo: el modelo diminuto va a cumplir el schema igual que el 4B, y aun así puede poner cualquier número en `dificultad`. Esa frontera es lo que separa esta sección de la ilusión de que structured outputs "arregla" al modelo.


In [31]:
from pydantic import BaseModel, Field

# El schema = una clase. Ojo con los campos de texto libre: si el modelo decide
# escribir un ensayo dentro de "tema", el JSON se corta por num_predict y NO parsea.
# Dos cinturones: el prompt pide brevedad, y max_length lo acota en el schema.
class Ficha(BaseModel):
    tema: str = Field(max_length=60)
    dificultad: int

P_FICHA = ("Generá una ficha para: {t}. "
           "El campo 'tema' es el NOMBRE del tema en 1 a 3 palabras, no una descripción. "
           "El campo 'dificultad' va de 1 (se explica en primaria) a 5 (nivel universitario "
           "avanzado); elegí el valor que corresponda, no siempre el mismo.")

# La prueba de fuego: el MISMO pedido con format=, en las mismas dos escalas.
# Si la garantía es real, el modelo diminuto tiene que cumplir igual que el 4B.
mostrar_ejemplo("STRUCTURED OUTPUTS (modelo diminuto) — prompt + respuesta",
                P_FICHA.format(t="fotosíntesis"),
                ask(P_FICHA.format(t="fotosíntesis"), fmt=Ficha.model_json_schema(),
                    model=MODEL_TINY))

filas_fmt, res_fmt = [], {}
for etiqueta, modelo in [("diminuto", MODEL_TINY), ("4B", MODEL)]:
    ok = 0
    for tema in TEMAS_JSON:
        raw = ask(P_FICHA.format(t=tema), fmt=Ficha.model_json_schema(), model=modelo)
        try:
            print(etiqueta.ljust(10), Ficha.model_validate_json(raw))
            ok += 1
        except Exception as e:
            print(etiqueta.ljust(10), "NO parseó:", repr(raw[:90]), "->", type(e).__name__)
    fila = {"cumplen el schema": "%d/%d" % (ok, len(TEMAS_JSON))}
    filas_fmt.append((etiqueta, fila))
    res_fmt[etiqueta] = fila["cumplen el schema"]

tabla("Con format=: ¿cumple el schema, y depende de la escala?",
      ["cumplen el schema"], filas_fmt)

# El resumen de cierre se arma con los números MEDIDOS, no escritos a mano.
RESULTADOS["json"] = {
    "a mano 4B (json.loads)":   RESULTADOS["json_a_mano"]["4B / limpio"]["json.loads"],
    "a mano chico (claves+tipos)": RESULTADOS["json_a_mano"]["diminuto / limpio"]["claves+tipos"],
    "con format= 4B":    res_fmt["4B"],
    "con format= chico": res_fmt["diminuto"],
}

──────────────────────────────────────────────────────────────────────
STRUCTURED OUTPUTS (modelo diminuto) — prompt + respuesta
──────────────────────────────────────────────────────────────────────
PROMPT:
Generá una ficha para: fotosíntesis. El campo 'tema' es el NOMBRE del tema en 1 a 3 palabras, no una descripción. El campo 'dificultad' va de 1 (se explica en primaria) a 5 (nivel universitario avanzado); elegí el valor que corresponda, no siempre el mismo.

RESPUESTA DEL MODELO:
{ "tema": "fotosíntesis", "dificultad": 5 }

diminuto   tema='fotosíntesis' dificultad=5
diminuto   tema='Derivadas' dificultad=5
diminuto   tema='Revolución Francesa' dificultad=5
4B         tema='Fotosíntesis' dificultad=3
4B         tema='Derivadas' dificultad=3
4B         tema='Revolución Francesa' dificultad=4

Con format=: ¿cumple el schema, y depende de la escala?
────────────────────────────────
               cumplen el schema
diminuto                     3/3
4B                           3/3



**La garantía — y lo que revela sobre la capacidad.** Con `format=` la **forma** está garantizada por construcción: cada campo, cada tipo, en un JSON bien armado, sin prosa alrededor.

Pero lo importante es mirar **quién** la cumple. El modelo diminuto —el mismo que colapsaba a una sola etiqueta en la Sección 3 y que en la Sección 5 no resuelve un problema de dos pasos— produce JSON que valida contra el schema igual que el 4B. **La restricción no depende de la capacidad del modelo, porque no vive en el modelo: vive en el decoder.**

Eso tiene una consecuencia práctica grande, y es el argumento real para usar structured outputs en producción: para extracción estructurada podés usar un modelo chico y barato **sin pagar nada en confiabilidad de formato**. Lo que sigue dependiendo de la capacidad es el **contenido** —si la dificultad que elige tiene sentido, si el tema está bien nombrado—, pero la forma deja de ser una fuente de riesgo.

Miremos primero qué le pasamos —el schema que genera Pydantic— y después la letra chica, porque hay algo que `format=` **no** garantiza.


In [32]:
import json
print(json.dumps(Ficha.model_json_schema(), indent=2, ensure_ascii=False))

{
  "properties": {
    "tema": {
      "maxLength": 60,
      "title": "Tema",
      "type": "string"
    },
    "dificultad": {
      "title": "Dificultad",
      "type": "integer"
    }
  },
  "required": [
    "tema",
    "dificultad"
  ],
  "title": "Ficha",
  "type": "object"
}


**La letra chica: la gramática está garantizada, la terminación no.** `format=` fuerza que *cada token emitido* mantenga la salida dentro del schema. Lo que no puede hacer es obligar al modelo a **terminar** antes de que se acabe `num_predict`. Si el modelo decide escribir un ensayo dentro de un campo `str`, el generador se corta a mitad de string y el JSON queda sin cerrar — sintácticamente inválido, pese al schema.

Reproduzcámoslo a propósito.


In [33]:
# Reproducimos el fallo A PROPÓSITO: campo de texto libre + presupuesto corto.
class FichaLibre(BaseModel):
    tema: str          # sin max_length: el modelo puede escribir lo que quiera
    resumen: str

p = ("Generá una ficha EXHAUSTIVA y detallada sobre la fotosíntesis, "
     "explayándote todo lo posible en cada campo.")
raw_libre = ask(p, fmt=FichaLibre.model_json_schema(), num_predict=120)

print("Salida cruda (cortada):")
print(raw_libre[-160:] if len(raw_libre) > 160 else raw_libre)
print()
try:
    FichaLibre.model_validate_json(raw_libre)
    print("parseó (subí el texto del prompt para forzar el corte)")
except Exception as e:
    print("NO parsea ->", str(e).splitlines()[0])
    print()
    print("Y sin embargo: mirá el final. No hay ni una llave de más, ni una comilla suelta,")
    print("ni un campo fuera del schema. Cada token emitido es válido. Lo que falta es el")
    print("FINAL: nos quedamos sin presupuesto antes de que el modelo cerrara el string.")
    print()
    print("La garantía es sobre la gramática, no sobre la terminación. En producción esto")
    print("se maneja acotando los campos (max_length), dando presupuesto suficiente, y")
    print("chequeando done_reason == 'length' — que es lo que hace nuestro ask().")

   [⚠ cortado por num_predict=120]
Salida cruda (cortada):
roceso es esencial para la vida en la Tierra, ya que proporciona alimento para los organismos autótrofos y libera oxígeno en la atmósfera. Esta ficha ofrece una

NO parsea -> 1 validation error for FichaLibre

Y sin embargo: mirá el final. No hay ni una llave de más, ni una comilla suelta,
ni un campo fuera del schema. Cada token emitido es válido. Lo que falta es el
FINAL: nos quedamos sin presupuesto antes de que el modelo cerrara el string.

La garantía es sobre la gramática, no sobre la terminación. En producción esto
se maneja acotando los campos (max_length), dando presupuesto suficiente, y
chequeando done_reason == 'length' — que es lo que hace nuestro ask().


**Lo que necesitábamos: una garantía, no una súplica.** Con la salvedad que acabamos de ver: la gramática es una garantía dura, el presupuesto de tokens sigue siendo tuyo.

La pregunta ahora es *cómo* se garantiza la forma en un modelo estocástico. Lo diseccionamos en la próxima sección.

▶ **Láminas 19–20, 26** (el problema; JSON mode vs structured outputs; la práctica con Ollama+Pydantic).


## 11 · El mecanismo, a mano: enmascarar logits (constrained decoding)

Acá está el mecanismo, corrido a mano con HuggingFace. En cada paso el modelo produce **logits sobre todo el vocabulario** (softmax, Clase 2). "Structured output" no hace que el modelo se esfuerce más: a los tokens que **romperían** la forma les ponemos el logit en **−∞** *antes* de elegir. La garantía vive en el **decoder**, no en el prompt.

**Qué mirar en la salida de abajo.** El resultado final es lo menos interesante de esta celda; lo que importa es lo que pasa en el medio. Tres cosas:

1. **Lo que el modelo *quería* decir.** Imprimimos su top-6 sin tocar nada. Vas a ver que ninguno de sus candidatos favoritos es una etiqueta válida: iba a empezar una oración.
2. **El puesto del token que terminó eligiendo.** Ésta es la línea que más dice. Si el elegido estaba en el puesto #40 de 50.257, significa que el modelo lo tenía prácticamente descartado y **la máscara lo volvió el ganador**. No hubo persuasión: hubo poda.
3. **Cuánta masa de probabilidad se apagó.** Qué porcentaje del total mandó la gramática a cero. Suele estar arriba del 99%.

> **Decisión didáctica:** usamos `gpt2` (chico y rápido). El punto no es la calidad del modelo — es el **mecanismo**. Y de hecho probablemente conteste **mal**: la reseña es negativa. Eso no es un defecto de la demo, es parte de la lección. La máscara garantiza que la salida sea *una etiqueta válida*, no que sea *la correcta* — y si mirás su ranking dentro de lo permitido, vas a ver que el error es del modelo, no de la restricción.


In [34]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
lm  = AutoModelForCausalLM.from_pretrained("gpt2").eval()

# Dos gramáticas para el MISMO mecanismo:
#  - TARGETS_LABEL: tres palabras. Simple — pero 'positive' es UN token en GPT-2, así que
#    el autómata decide una sola vez y no se lo ve avanzar.
#  - TARGETS_JSON: un mini-JSON. Varios tokens, así que el abanico de tokens válidos se
#    puede VER abriéndose y cerrándose paso a paso (es la lámina 23, con números).
TARGETS_LABEL = ["positive", "negative", "neutral"]
TARGETS_JSON  = ['{"sent": "pos"}', '{"sent": "neg"}']

# Precomputamos el texto de cada token del vocabulario UNA vez.
id2str = [tok.decode([i]) for i in range(lm.config.vocab_size)]
print("Vocabulario:", len(id2str), "tokens")

Vocabulario: 50257 tokens


In [35]:
def allowed_ids(generated, targets):
    """Tokens cuyo texto mantiene 'generated' como prefijo (o completa) algún target."""
    out = []
    for i, cand in enumerate(id2str):
        norm = (generated + cand).lstrip()          # tolera el espacio inicial del BPE
        if not norm:
            # Token de puro whitespace: sin este guard, t.startswith("") da True para
            # todos los targets y el decoder puede quedar acumulando espacios.
            continue
        if any(t == norm or t.startswith(norm) for t in targets):
            out.append(i)
    return out

def un_paso(ids, generated, targets, k=6):
    """UN paso de decoding, devolviendo todo lo que pasa por el medio —que es el punto
    de esta sección— en vez de sólo el token elegido."""
    with torch.no_grad():
        logits = lm(ids).logits[0, -1]              # (vocab,) logits del próximo token
    probs = torch.softmax(logits, dim=-1)           # lo que el modelo QUERÍA, sin tocar nada
    allow = allowed_ids(generated, targets)
    if not allow:
        return None
    masked = torch.full_like(logits, float("-inf"))
    masked[allow] = logits[allow]                   # sólo los válidos conservan su logit
    nxt = int(torch.argmax(masked))                 # elegimos DENTRO de lo válido
    return {
        "allow": allow,
        "nxt": nxt,
        # guardamos IDS (no el texto): asi el display consulta el MISMO conjunto
        # que aplica la mascara, y no puede divergir de el.
        "libre":   [(int(i), float(p)) for p, i in zip(*torch.topk(probs, k))],
        "masa":    float(probs[allow].sum()),        # masa que sobrevive a la máscara
        "rank":    int((probs > probs[nxt]).sum()) + 1,   # puesto GLOBAL del elegido
        "p_libre": float(probs[nxt]),                # su prob. antes de enmascarar
        "p_mask":  float(torch.softmax(masked, dim=-1)[nxt]),   # y después
    }

def mostrar_paso(step, generado, p, targets):
    permitidos = set(p["allow"])   # el conjunto REAL que aplica la mascara
    print("=" * 74)
    print("PASO %d   |   generado hasta ahora: %s" % (step, repr(generado)))
    print("=" * 74)
    print("Lo que el modelo QUERÍA decir (su top %d, sin restricción):" % len(p["libre"]))
    for tid, prob in p["libre"]:
        print("     %-16s p=%.4f%s" % (repr(id2str[tid]), prob,
                                        "   <- es válido" if tid in permitidos else ""))
    print()
    print("Tokens PERMITIDOS por la gramática: %d de %d   (%.3f%% del vocabulario)"
          % (len(p["allow"]), len(id2str), 100 * len(p["allow"]) / len(id2str)))
    print("     " + ", ".join(repr(id2str[i]) for i in p["allow"][:14])
          + (" ..." if len(p["allow"]) > 14 else ""))
    print()
    print("Masa de probabilidad que SOBREVIVE a la máscara: %.2f%%  (se apagó el %.2f%%)"
          % (100 * p["masa"], 100 * (1 - p["masa"])))
    print("Elegido: %s" % repr(id2str[p["nxt"]]))
    print("     era el puesto #%d de %d en el ranking del modelo (p=%.5f)"
          % (p["rank"], len(id2str), p["p_libre"]))
    print("     con la máscara pasa a p=%.3f" % p["p_mask"])
    print()

def constrained_decode(prompt, targets, max_steps=14, detalle=0):
    ids, generated, traza = tok(prompt, return_tensors="pt").input_ids, "", []
    for step in range(max_steps):
        p = un_paso(ids, generated, targets)
        if p is None:
            break
        if step < detalle:
            mostrar_paso(step, generated, p, targets)
        generated += id2str[p["nxt"]]
        traza.append((step, len(p["allow"]), id2str[p["nxt"]], generated))
        ids = torch.cat([ids, torch.tensor([[p["nxt"]]])], dim=1)
        if generated.lstrip() in targets:
            break
    return generated.strip(), traza

prompt_gpt2 = "Review: 'The plot was boring and I fell asleep.' Sentiment (one word):"
print("PROMPT:", repr(prompt_gpt2), "\n")
salida_label, _ = constrained_decode(prompt_gpt2, TARGETS_LABEL, detalle=1)
print("CON restricción ->", repr(salida_label))

PROMPT: "Review: 'The plot was boring and I fell asleep.' Sentiment (one word):" 

PASO 0   |   generado hasta ahora: ''
Lo que el modelo QUERÍA decir (su top 6, sin restricción):
     " '"             p=0.5890
     ' "'             p=0.1105
     ' I'             p=0.0144
     '\n'             p=0.0123
     ' The'           p=0.0066
     ' A'             p=0.0056

Tokens PERMITIDOS por la gramática: 20 de 50257   (0.040% del vocabulario)
     'n', 'p', ' p', ' n', ' ne', 'ne', ' po', ' pos', 'pos', ' neg', ' positive', ' negative', 'po', ' neutral' ...

Masa de probabilidad que SOBREVIVE a la máscara: 0.04%  (se apagó el 99.96%)
Elegido: ' p'
     era el puesto #294 de 50257 en el ranking del modelo (p=0.00014)
     con la máscara pasa a p=0.333

CON restricción -> 'positive'


In [36]:
# Sin restricción, el mismo modelo produce texto arbitrario (no necesariamente un label).
ids_gpt2 = tok(prompt_gpt2, return_tensors="pt").input_ids
with torch.no_grad():
    out = lm.generate(ids_gpt2, max_new_tokens=6, do_sample=False,
                      pad_token_id=tok.eos_token_id)
print("SIN restricción ->", repr(tok.decode(out[0, ids_gpt2.shape[1]:])))

SIN restricción -> " 'I'm not sure what"


**El abanico, ahora sí visible.** La celda de arriba tiene una limitación: `' positive'` es **un solo token** en el vocabulario de GPT-2. El autómata elige una vez y termina, así que nunca se lo ve avanzar de estado.

Para verlo caminar hace falta un target de varios tokens. Usamos un mini-JSON (`{"sent": "pos"}`), que encima se parece mucho más a lo que structured outputs hace de verdad. Mirá la columna **permitidos**: es el tamaño del conjunto de tokens válidos en cada estado de la gramática.

Es la lámina 23 —el mecanismo caminado— pero con los números de tu corrida en lugar de un diagrama.


In [37]:
# El mismo mecanismo, con un target de VARIOS tokens: ahora el autómata tiene que
# caminar, y el abanico de tokens válidos se puede ver en cada estado.
prompt_json = "Review: 'The plot was boring and I fell asleep.'\nSentiment as JSON: "
salida_json, traza = constrained_decode(prompt_json, TARGETS_JSON)

print("PROMPT:", repr(prompt_json))
print("CON restricción ->", repr(salida_json), "\n")

print("El abanico, paso a paso (la gramática avanzando de estado):")
print("paso".rjust(4), "permitidos".rjust(11), " ", "elegido".ljust(12), "generado hasta ahora")
print("-" * 74)
for step, n, elegido, acum in traza:
    print(str(step).rjust(4), str(n).rjust(11), " ", repr(elegido).ljust(12), repr(acum))

print()
print("Fijate que el número NO baja siempre: cuando la gramática llega a un punto con")
print("varias continuaciones válidas (espacios, comillas, variantes de BPE del mismo")
print("texto) el abanico se vuelve a abrir. Eso es el autómata cambiando de estado.")

PROMPT: "Review: 'The plot was boring and I fell asleep.'\nSentiment as JSON: "
CON restricción -> '{"sent": "neg"}' 

El abanico, paso a paso (la gramática avanzando de estado):
paso  permitidos   elegido      generado hasta ahora
--------------------------------------------------------------------------
   0           4   '{'          '{'
   1           1   '"'          '{"'
   2           4   'sent'       '{"sent'
   3           2   '":'         '{"sent":'
   4           2   ' "'         '{"sent": "'
   5           6   'n'          '{"sent": "n'
   6           2   'eg'         '{"sent": "neg'
   7           2   '"'          '{"sent": "neg"'
   8           1   '}'          '{"sent": "neg"}'

Fijate que el número NO baja siempre: cuando la gramática llega a un punto con
varias continuaciones válidas (espacios, comillas, variantes de BPE del mismo
texto) el abanico se vuelve a abrir. Eso es el autómata cambiando de estado.


**El Aha!, en código.** El JSON (o el enum) garantizado no es persuasión: es una **máscara sobre la distribución** en el decoder. El modelo sigue eligiendo —hay libertad *dentro* de lo válido—, pero fuera de la gramática la probabilidad es exactamente cero.

Y ahora tenés los tres números que lo demuestran, en vez de tener que creerlo: el modelo quería decir otra cosa, el token que emitió estaba enterrado en su propio ranking, y la gramática apagó casi toda la masa de probabilidad. **Nada de eso se consigue escribiendo un prompt mejor**, porque no es un problema de prompt: es una operación sobre la distribución que el prompt ya produjo.

> **Nota sobre el costo.** Recorrimos las ~50.000 entradas del vocabulario en **cada** paso (O(vocab)) — y con el target de JSON eso pasó varias veces. Es exactamente lo que los sistemas reales evitan **precomputando un índice** estado → tokens válidos. Por qué eso no es trivial, en la próxima sección.

▶ **Láminas 21–23** (la distribución sobre el vocabulario; enmascarar logits; el mecanismo caminado).


## 12 · Por qué no es trivial: token vs carácter

La gramática está a nivel de **caracteres** (`{`, `"`, un dígito), pero el modelo emite **sub-word tokens**. Un solo token puede ser `" name"` —un espacio más letras— y cruzar **varios** estados de la gramática de una. Mirémoslo con el tokenizer sobre un JSON.


In [38]:
ejemplo = '{"name": "Ada"}'
ids_demo = tok(ejemplo).input_ids
print("Texto:", ejemplo)
print("Tokens (uno por línea):")
for i in ids_demo:
    print("   id", i, "->", repr(tok.decode([i])))

Texto: {"name": "Ada"}
Tokens (uno por línea):
   id 4895 -> '{"'
   id 3672 -> 'name'
   id 1298 -> '":'
   id 366 -> ' "'
   id 2782 -> 'Ad'
   id 64 -> 'a'
   id 20662 -> '"}'


**El desajuste, a la vista.** Varios tokens contienen **espacio + letras** o **símbolo + comilla** juntos: no coinciden con las fronteras de la gramática, que son por carácter.

Contalos en la salida de arriba: de siete tokens, **cuatro cruzan una frontera estructural**.

- `'{"'` abre el objeto **y** abre el string de la clave → dos estados de una
- `'":'` cierra el string de la clave **y** consume los dos puntos
- `' "'` se come el espacio opcional de JSON **junto con** la comilla de apertura del valor
- `'"}'` cierra el string **y** cierra el objeto

Por eso *"¿este carácter es válido?"* es la pregunta equivocada — no elegís caracteres, elegís tokens. La pregunta correcta es bastante más incómoda: **dado que estoy en el estado q, ¿qué tokens puedo emitir tal que *todos* sus caracteres me mantengan dentro de la gramática, y en qué estado quedo después?**

Es el mismo problema token-vs-carácter de la clase de tokenización, ahora del lado de la generación. Y responder esa pregunta rápido es lo único que hace que structured outputs sea barato — así que veamos cómo se hace.


**Cómo se hace barato: precomputar el índice.**

La celda de la Sección 11 resolvió el problema por fuerza bruta. Mirá `allowed_ids`: recorre las ~50.000 entradas del vocabulario **en cada paso** para ver cuáles encajan. Sirve para una demo; en producción, no. El truco real se entiende mejor con un vocabulario de juguete.

**La gramática:** la salida tiene que ser exactamente `true` o `false`.
**El vocabulario:** diez tokens, en vez de 50.257.

```
id 0: 't'       id 5: 'f'
id 1: 'tr'      id 6: 'fa'
id 2: 'true'    id 7: 'false'
id 3: 'rue'     id 8: 'alse'
id 4: 'ue'      id 9: 'lse'
```

**La forma bruta.** Ya escribiste `"tr"` y querés saber qué podés emitir ahora. Probás los diez, uno por uno:

```
't'     → "trt"       ¿es prefijo de true/false?   no
'tr'    → "trtr"                                   no
'true'  → "trtrue"                                 no
'rue'   → "trrue"                                  no
'ue'    → "true"                                   SÍ  ✓
'f'     → "trf"                                    no
'fa'    → "trfa"                                   no
'false' → "trfalse"                                no
'alse'  → "tralse"                                 no
'lse'   → "trlse"                                  no
```

Diez pruebas para decidir **un** token. Con el vocabulario real son 50.257 — y en el paso siguiente se repite todo desde cero, y en el próximo pedido también.

**La observación que lo cambia todo.** El resultado de esa prueba **no depende del prompt, ni del modelo, ni de las probabilidades.** Depende de dos cosas nada más: en qué estado estás y cuál es el vocabulario. Así que se puede calcular de antemano, para todos los estados posibles, y guardarlo en una tabla:

| Estado (lo escrito) | Tokens permitidos | Te dejan en |
|---|---|---|
| `""` | `'t'` · `'tr'` · `'true'` · `'f'` · `'fa'` · `'false'` | `"t"` · `"tr"` · **fin** · `"f"` · `"fa"` · **fin** |
| `"t"` | `'rue'` | **fin** |
| `"tr"` | `'ue'` | **fin** |
| `"f"` | `'alse'` | **fin** |
| `"fa"` | `'lse'` | **fin** |

**Esa tabla es el índice.** No es más que un diccionario: para cada estado, qué tokens son válidos y a qué estado te mueve cada uno. Con la tabla armada, generar pasa a ser una consulta:

```
estoy en "tr"   →   miro la fila   →   permitidos = {'ue'}
```

**El truco, en una frase: es el mismo bucle, movido de adentro para afuera.**

```python
# FUERZA BRUTA (lo que hace esta notebook): el bucle vive ADENTRO de la generación
for paso in range(...):                 # por cada token que generás
    for tid in range(50257):            #   recorrés todo el vocabulario... otra vez
        if encaja(estado, tid): ...

# CON ÍNDICE: el mismo bucle vive AFUERA, y corre UNA sola vez
for estado in estados:                  # antes de generar nada
    for tid in range(50257):            #   recorrés todo el vocabulario, y lo anotás
        if encaja(estado, tid):
            INDICE[estado].append(tid)

for paso in range(...):                 # por cada token que generás
    permitidos = INDICE[estado]         #   una consulta: no recorrés nada
```

Ojo con esto, porque es lo que más se malentiende: **el trabajo total no baja.** Seguís probando todos los tokens contra todos los estados. Lo que cambia es *cuándo* lo hacés — una vez al principio, en vez de en cada paso de cada pedido.

**Y de ahí salen los dos costos tan distintos** de los que habla la lámina 25:

| | Cuándo | Cuánto cuesta | ¿Se repite? |
|---|---|---|---|
| Compilar el índice | una vez por schema | algo | no: se **cachea** |
| Enmascarar | por cada token generado | una consulta a un diccionario | sí, pero es despreciable |

La primera llamada con un schema paga la construcción. La segunda, la milésima y la millonésima reusan el índice cacheado. Por eso restringir sale "casi gratis": el costo real es el forward pass del modelo, no la máscara.

Es el mismo mecanismo que el índice al final de un libro. Construirlo cuesta leer el libro entero una vez; después buscar un tema son dos segundos, en lugar de hojear 400 páginas cada vez que querés buscar algo.

> **Una simplificación del ejemplo.** Acá usé como clave *"el texto escrito hasta ahora"*, que en este juguete coincide con el estado del autómata. Las implementaciones reales indexan por **estado**, que es mucho más compacto: para un JSON cualquiera hay infinitos prefijos posibles, pero un puñado de estados. La idea es idéntica; cambia sólo qué usás como clave del diccionario.

▶ **Lámina 24** (la conexión con tokenización; el índice de Outlines).


## 13 · Tres verificaciones

Tres cosas que el deck afirma y que acá comprobamos: (a) el schema **no se inyecta** en el prompt; (b) los **nombres/orden de los campos** mueven la accuracy; (c) para tareas verificables medimos con un **verificador duro**, no con el juez.


**(a) El schema no se inyecta en el prompt.** El texto que lee el modelo es idéntico con y sin structured outputs; el schema viaja en el kwarg `format=`, es decir en el **decoder**, no en lo que el modelo lee.


In [39]:
p = "Decime el sentimiento de: 'Me encantó, la volvería a comprar'."
class S(BaseModel):
    sentimiento: str

sin = ask(p)                              # sin structured outputs
con = ask(p, fmt=S.model_json_schema())   # con structured outputs
print("Prompt enviado (idéntico en ambos casos):")
print("  ", repr(p))
print("El schema NO está en el prompt: va en format=. Sólo cambia el decoder.")
print("\nSalida sin structured:", repr(sin))
print("Salida con structured:", repr(con))

Prompt enviado (idéntico en ambos casos):
   "Decime el sentimiento de: 'Me encantó, la volvería a comprar'."
El schema NO está en el prompt: va en format=. Sólo cambia el decoder.

Salida sin structured: 'El sentimiento de la frase **"Me encantó, la volvería a comprar"** es **positivo**.\n\nEsta expresión muestra satisfacción y entusiasmo por un producto o experiencia, ya que la persona no solo dice que le gustó mucho, sino que incluso estaría dispuesta a comprarlo de nuevo. Esto indica una opinión muy favorable y un alto grado de satisfacción.'
Salida con structured: '{ "sentimiento": "positivo" }'


**(b) Los nombres de los campos son parte del prompt.** Las keys las **genera el modelo, como tokens**: lo que aparece primero en la salida se computa primero. Poner un campo de razonamiento *antes* de la respuesta le devuelve al modelo el lugar para pensar que la restricción, mal usada, le saca.

Ahora bien: para comparar "razonamiento primero" contra "respuesta primero" hay que poder **controlar el orden**. Y ahí hay una sorpresa del backend que conviene ver antes de diseñar el experimento.


**Antes del experimento, una sorpresa.** Vamos a comparar dos schemas que se diferencian en el **orden** de los campos. Pero, ¿respeta el backend el orden en que los declaramos? Preguntémosle en vez de suponerlo: declaramos `zeta` primero y `alfa` segundo, y miramos qué sale.


In [40]:
class OrdenDeclarado(BaseModel):
    zeta: str      # declarado PRIMERO
    alfa: str      # declarado SEGUNDO

raw_orden = ask("Devolveme un JSON con zeta='Z' y alfa='A'.",
                fmt=OrdenDeclarado.model_json_schema())

print("Declaramos:  zeta, alfa")
print("Salió     : ", raw_orden.replace("\n", " "))
print()
claves = [k for k in ["zeta", "alfa"] if k in raw_orden]
orden_ok = raw_orden.find("zeta") < raw_orden.find("alfa")
print("¿Respetó el orden declarado?", "SÍ" if orden_ok else "NO — salió en orden ALFABÉTICO")
print()
print("Ollama (llama.cpp) serializa las claves ordenadas alfabéticamente, ignorando el")
print("orden del schema. Consecuencia para lo que sigue: NO alcanza con reordenar los")
print("campos en la clase de Pydantic. Para controlar qué se genera primero hay que")
print("elegir los NOMBRES. Es un detalle de implementación que cambia el diseño del")
print("schema — y es exactamente el tipo de cosa que sólo aparece si mirás la salida cruda.")

Declaramos:  zeta, alfa
Salió     :  {   "alfa": "A",   "zeta": "Z" }

¿Respetó el orden declarado? NO — salió en orden ALFABÉTICO

Ollama (llama.cpp) serializa las claves ordenadas alfabéticamente, ignorando el
orden del schema. Consecuencia para lo que sigue: NO alcanza con reordenar los
campos en la clase de Pydantic. Para controlar qué se genera primero hay que
elegir los NOMBRES. Es un detalle de implementación que cambia el diseño del
schema — y es exactamente el tipo de cosa que sólo aparece si mirás la salida cruda.


In [41]:
from typing import Literal

# Dos regímenes. Como el backend ordena alfabéticamente, elegimos NOMBRES que produzcan
# el orden que queremos estudiar. Los nombres dejan de ser cosméticos: son la palanca.
class RazonPrimero(BaseModel):      # analisis < sentimiento  -> razona ANTES de decidir
    analisis: str
    sentimiento: Literal["positivo", "negativo", "neutral"]

class RespuestaPrimero(BaseModel):  # etiqueta < justificacion -> se compromete PRIMERO
    etiqueta: Literal["positivo", "negativo", "neutral"]
    justificacion: str

CAMPO_LABEL = {RazonPrimero: "sentimiento", RespuestaPrimero: "etiqueta"}

def build_p13(texto):
    return "Clasificá el sentimiento del texto.\nTexto: " + texto

def clasificar_struct(texto, schema, modelo_llm=MODEL):
    raw = ask(build_p13(texto), fmt=schema.model_json_schema(), model=modelo_llm)
    try:
        return raw, getattr(schema.model_validate_json(raw), CAMPO_LABEL[schema])
    except Exception:
        return raw, "?"

# Ejemplo: MISMO texto, dos schemas. Mirá con qué campo ARRANCA cada salida cruda.
t0 = EVAL_SENT[0][0]
raw_rp, _ = clasificar_struct(t0, RazonPrimero)
raw_pr, _ = clasificar_struct(t0, RespuestaPrimero)
mostrar_ejemplo("RAZONAMIENTO-PRIMERO (analisis, sentimiento)", build_p13(t0), raw_rp)
mostrar_ejemplo("RESPUESTA-PRIMERO (etiqueta, justificacion)",  build_p13(t0), raw_pr)

def acc_struct(schema, modelo_llm):
    preds = [clasificar_struct(t, schema, modelo_llm)[1] for t, _ in EVAL_SENT]
    golds = [g for _, g in EVAL_SENT]
    return round(sum(p == g for p, g in zip(preds, golds)) / len(golds), 2)

# Dos escalas: con el 4B esta tarea tiene techo, así que un empate no significa
# "el orden no importa", significa "no queda margen para que importe".
filas13 = []
for nombre, modelo_llm in ESCALAS:
    filas13.append((nombre, {"razonamiento primero": acc_struct(RazonPrimero, modelo_llm),
                             "respuesta primero":    acc_struct(RespuestaPrimero, modelo_llm)}))

tabla("Orden de los campos, por escala (n = " + str(len(EVAL_SENT)) + ")",
      ["razonamiento primero", "respuesta primero"], filas13)

RESULTADOS["orden_campos"] = dict(filas13)

──────────────────────────────────────────────────────────────────────
RAZONAMIENTO-PRIMERO (analisis, sentimiento)
──────────────────────────────────────────────────────────────────────
PROMPT:
Clasificá el sentimiento del texto.
Texto: La comida estaba fría y el servicio, pésimo.

RESPUESTA DEL MODELO:
{"analisis": "El texto menciona que la comida estaba fría y que el servicio fue pésimo. Ambos elementos indican una experiencia negativa. La palabra 'pésimo' es un adjetivo con un tono claramente negativo, lo que refuerza la percepción de descontento. Por lo tanto, el sentimiento general del texto es negativo.","sentimiento": "negativo"}

──────────────────────────────────────────────────────────────────────
RESPUESTA-PRIMERO (etiqueta, justificacion)
──────────────────────────────────────────────────────────────────────
PROMPT:
Clasificá el sentimiento del texto.
Texto: La comida estaba fría y el servicio, pésimo.

RESPUESTA DEL MODELO:
{"etiqueta": "negativo", "justificacion": "El te

**Lo que acabás de ver.** La tabla tiene dos filas, y conviene leerlas sin decidir de antemano qué tiene que pasar.

La hipótesis de *reason-then-structure* es que **"razonamiento primero" le gane a "respuesta primero"**: el campo de análisis le da al modelo tokens donde computar *antes* de comprometerse con una etiqueta. Es la misma física de la Sección 5, ahora dentro de un JSON.

Pero mirá los números antes de darla por buena. En el **4B** la tarea tiene techo: si las dos columnas empatan, eso no significa "el orden no importa", significa que no quedaba margen para que importara. Y en la escala chica —donde sí hay margen— es perfectamente posible que el resultado salga **al revés**, con "respuesta primero" ganando. Si eso pasó en tu corrida, hay dos lecturas y las dos valen:

- **Ruido.** Son 12 casos: dos casos de diferencia son 0.17 de accuracy. A este tamaño de muestra, una inversión no es evidencia de nada. Es el mismo criterio de la Sección 3.
- **El efecto es real pero chico, y depende de la tarea.** Clasificar sentimiento en una frase corta no necesita gran deliberación; el campo de análisis agrega tokens que no hacen falta y puede introducir ruido. Reason-then-structure paga en tareas donde la respuesta *requiere* derivación, no en las que el modelo resuelve de una.

Que un resultado publicado no se reproduzca en tu setup es información, no un fracaso del experimento. La conclusión honesta acá es **"a esta escala y en esta tarea, el orden de los campos no movió la aguja"** — y eso es distinto de "el orden nunca importa".

Lo que sí queda firme, independientemente del número, es el hallazgo de la celda anterior: como el orden de generación lo decide el **orden alfabético de los nombres**, elegir cómo se llama un campo deja de ser una cuestión de estilo. `analisis` antes que `sentimiento` sale en ese orden; `razonamiento` después de `etiqueta` no, por más que lo declares primero en la clase. Si diseñás un schema para que el modelo razone antes de responder, tenés que **nombrar los campos para que el alfabeto te acompañe** — o verificar qué hace tu backend, porque esto no está estandarizado.


**(c) Verificable (duro) vs juez (blando).** Para una tarea con respuesta verificable —extraer los enteros de un texto a un JSON— el chequeo es **programático**, no subjetivo. Ese contraste es el mapa de la Clase 6: verificador **duro** = RLVR; juez LLM **blando** = RLHF.


In [42]:
class Numeros(BaseModel):
    numeros: list[int]

texto = "Compré 3 manzanas y 12 bananas, y pagué 250 pesos."
raw = ask("Extraé todos los enteros del texto a un JSON.\nTexto: " + texto,
          fmt=Numeros.model_json_schema())
print("salida:", raw)

# Verificador DURO (programático): ¿parsea y tipa? No hace falta juez.
try:
    obj = Numeros.model_validate_json(raw)
    print("verificador programático (duro):", True, "->", obj.numeros)
except Exception as e:
    print("verificador programático (duro):", False, "->", e)

salida: {
  "numeros": [
    3,
    12,
    250
  ]
}
verificador programático (duro): True -> [3, 12, 250]


**El puente que reúne las dos palancas.** Expandir (razonamiento) y restringir (forma) operan sobre la **misma** capacidad de generación: por eso "razonamiento antes que respuesta" funciona, y por eso *capacidad vs forma* es el mismo eje de la Clase 6. Y el debate del deck (¿restringir degrada el razonamiento?) se resuelve ahí: depende de la capacidad sobrante.

▶ **Láminas 25–28** (el costo; la práctica; los nombres de campos; el debate *Let Me Speak Freely* → *Capacity, Not Format*).


## 14 · Function calling: structured outputs con un propósito

Function calling **es** structured output: la forma que se fuerza es la de una **llamada a función** (nombre + argumentos tipados). El modelo no ejecuta nada — emite una intención estructurada; nuestro código la ejecuta y le devuelve el resultado. Lo corremos con el cliente nativo de Ollama (deja ver los `tool_calls`) y después miramos cómo queda el prompt con el `chat_template` de HuggingFace.


In [43]:
# 1) Una función Python real y su schema (mismo formato de tool que usa OpenAI/Ollama).
def clima(ciudad):
    datos = {"buenos aires": "22°C y soleado",
             "madrid": "14°C y nublado",
             "tokio": "9°C con lluvia"}
    return datos.get(ciudad.lower(), "sin datos")

tools = [{
    "type": "function",
    "function": {
        "name": "clima",
        "description": "Devuelve el clima actual de una ciudad.",
        "parameters": {
            "type": "object",
            "properties": {"ciudad": {"type": "string", "description": "Nombre de la ciudad"}},
            "required": ["ciudad"],
        },
    },
}]

def mostrar_conversacion(msgs, titulo):
    """Imprime la lista de mensajes. ESA lista es el estado de la conversación: crece en
    cada vuelta del loop y es lo único que el modelo ve en la llamada siguiente."""
    print("-" * 72)
    print(titulo + "   ->   " + str(len(msgs)) + " mensaje(s) en la lista")
    print("-" * 72)
    for i, m in enumerate(msgs):
        # los mensajes pueden ser dicts nuestros U objetos Message que devuelve ollama
        d = m if isinstance(m, dict) else {
            "role":       getattr(m, "role", None),
            "content":    getattr(m, "content", None),
            "tool_calls": getattr(m, "tool_calls", None),
            "tool_name":  getattr(m, "tool_name", None)}
        print("[%d] %s" % (i, str(d.get("role")).upper()))
        cont = d.get("content")
        if cont:
            print("     content  :", repr(cont[:150] + ("..." if len(cont) > 150 else "")))
        else:
            print("     content  : (vacío)")
        for c in (d.get("tool_calls") or []):
            print("     tool_call:", c.function.name + "(" + str(dict(c.function.arguments)) + ")")
        if d.get("tool_name"):
            print("     tool_name:", d.get("tool_name"))
    print()

# 2) Primera llamada: el modelo decide si llama a la herramienta.
msgs = [{"role": "user", "content": "¿Qué clima hace en Tokio? Respondé en una oración."}]
mostrar_conversacion(msgs, "PASO 1 · arrancamos con el turno del usuario")

print("Ojo: `tools` NO aparece en la lista de mensajes — viaja como kwarg aparte, igual")
print("que `format=`. Pero a diferencia de `format=`, SÍ termina dentro del prompt:")
print("lo vemos dos celdas más abajo, con el chat_template.\n")

r = ollama.chat(model=MODEL, messages=msgs, tools=tools, think=False)
print("¿tool_calls?:", r.message.tool_calls)
print("¿content?   :", repr(r.message.content),
      "<- vacío: emitió una INTENCIÓN, no texto")

------------------------------------------------------------------------
PASO 1 · arrancamos con el turno del usuario   ->   1 mensaje(s) en la lista
------------------------------------------------------------------------
[0] USER
     content  : '¿Qué clima hace en Tokio? Respondé en una oración.'

Ojo: `tools` NO aparece en la lista de mensajes — viaja como kwarg aparte, igual
que `format=`. Pero a diferencia de `format=`, SÍ termina dentro del prompt:
lo vemos dos celdas más abajo, con el chat_template.

¿tool_calls?: [ToolCall(function=Function(name='clima', arguments={'ciudad': 'Tokio'}))]
¿content?   : '' <- vacío: emitió una INTENCIÓN, no texto


In [44]:
# 3) Ejecutamos la herramienta y le devolvemos el resultado al modelo.
if r.message.tool_calls:
    msgs.append(r.message)                        # el turno del asistente (con la llamada)
    mostrar_conversacion(msgs, "PASO 2 · el modelo PIDIÓ la herramienta (nadie la ejecutó aún)")

    for call in r.message.tool_calls:
        nombre = call.function.name
        args   = dict(call.function.arguments)
        resultado = clima(**args) if nombre == "clima" else "sin datos"
        print(">>> nuestro código ejecuta:", nombre, args, "->", resultado, "\n")
        msgs.append({"role": "tool", "content": resultado, "tool_name": nombre})
    mostrar_conversacion(msgs, "PASO 3 · nuestro código ejecutó y devolvió el resultado")

    final = ollama.chat(model=MODEL, messages=msgs, think=False)
    msgs.append(final.message)                    # lo agregamos para ver la conversación completa
    mostrar_conversacion(msgs, "PASO 4 · el modelo responde apoyándose en el resultado")

    print("Respuesta final:", final.message.content)
    print()
    print("Esa lista de 4 mensajes ES el estado del agente. Fijate el reparto de trabajo:")
    print("  [1] el modelo PIDE          (content vacío + tool_call)")
    print("  [2] nuestro código HACE     (role='tool' con el resultado)")
    print("  [3] el modelo REDACTA       (recién acá hay texto para el usuario)")
    print()
    print("En un agente real el bucle sigue, y cada vuelta agrega DOS mensajes: la")
    print("intención y el resultado. El contexto crece solo — que es exactamente el")
    print("problema que medimos en la Sección 15.")
else:
    print("El modelo respondió sin usar la herramienta:", r.message.content)

------------------------------------------------------------------------
PASO 2 · el modelo PIDIÓ la herramienta (nadie la ejecutó aún)   ->   2 mensaje(s) en la lista
------------------------------------------------------------------------
[0] USER
     content  : '¿Qué clima hace en Tokio? Respondé en una oración.'
[1] ASSISTANT
     content  : (vacío)
     tool_call: clima({'ciudad': 'Tokio'})

>>> nuestro código ejecuta: clima {'ciudad': 'Tokio'} -> 9°C con lluvia 

------------------------------------------------------------------------
PASO 3 · nuestro código ejecutó y devolvió el resultado   ->   3 mensaje(s) en la lista
------------------------------------------------------------------------
[0] USER
     content  : '¿Qué clima hace en Tokio? Respondé en una oración.'
[1] ASSISTANT
     content  : (vacío)
     tool_call: clima({'ciudad': 'Tokio'})
[2] TOOL
     content  : '9°C con lluvia'
     tool_name: clima

-------------------------------------------------------------------

**Dónde vive el schema de la herramienta.** El `chat_template` del tokenizer inyecta las definiciones de tools en el prompt en un formato fijo. Miralo con HuggingFace: así es como el modelo "ve" las herramientas disponibles.


In [45]:
from transformers import AutoTokenizer

# El chat_template tiene que ser el de la familia que estamos corriendo: el formato de
# tools cambió entre Qwen2.5 y Qwen3, así que mostrar el de Qwen2.5 sería enseñar algo que
# NO es lo que este modelo ve. Probamos repos en orden y usamos el primero que resuelva
# (con fallback: esta demo no puede morir en vivo).
CANDIDATOS_HF = ["Qwen/Qwen3-4B-Instruct-2507", "Qwen/Qwen3-4B", "Qwen/Qwen2.5-0.5B-Instruct"]

tk, repo_usado = None, None
for repo in CANDIDATOS_HF:
    try:
        tk, repo_usado = AutoTokenizer.from_pretrained(repo), repo
        break
    except Exception as e:
        print("no pude cargar", repo, "->", type(e).__name__)

print("tokenizer:", repo_usado, "\n")
prompt_con_tools = tk.apply_chat_template(
    [{"role": "user", "content": "¿Qué clima hace en Tokio?"}],
    tools=tools, add_generation_prompt=True, tokenize=False,
)
print(prompt_con_tools)

tokenizer: Qwen/Qwen3-4B-Instruct-2507 

<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "clima", "description": "Devuelve el clima actual de una ciudad.", "parameters": {"type": "object", "properties": {"ciudad": {"type": "string", "description": "Nombre de la ciudad"}}, "required": ["ciudad"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
¿Qué clima hace en Tokio?<|im_end|>
<|im_start|>assistant



**El cierre del arco de forma.** Few-shot condiciona; structured output restringe; function calling restringe **hacia una acción**. Es la misma palanca —moldear la distribución del próximo token— apuntada a integrarse con el mundo. Y es el ladrillo de los **agentes** (Unidad 4).

> **Ojo con un detalle (del deck):** demasiadas tools confunden al modelo. Menos herramientas, bien descriptas, rinden mejor — se conecta con *context engineering*, dos secciones más abajo.

▶ **Láminas 29–30** (function calling como structured output; el diseño de tools).


## 15 · Lost in the Middle y Context Rot

Más contexto no es gratis ni uniforme. **Lost in the Middle**: lo que está en el **medio** del contexto se recupera peor que lo que está al principio o al final (curva en U). **Context Rot**: a medida que el contexto se alarga, el rendimiento **cae** aunque la tarea sea igual de simple. Lo reproducimos: escondemos un dato ("needle") en distintas posiciones y medimos si el modelo lo encuentra.


In [46]:
temas = ["el clima de la Patagonia", "la historia del café", "las mareas oceánicas",
         "la fotografía analógica", "los volcanes de Islandia", "el ajedrez postal",
         "la migración de las ballenas", "la música barroca", "los faros del Atlántico",
         "la apicultura urbana", "las rutas de la seda", "los relojes de arena",
         "la cartografía medieval", "los jardines japoneses", "la cría de caracoles",
         "las auroras boreales", "los molinos de viento", "la caligrafía china",
         "los puentes colgantes", "las cometas tradicionales"]

# Cada distractor lleva SU PROPIO código, con la misma pinta que el que buscamos.
# Por qué: si el needle es el único texto del contexto que menciona un código, encontrarlo
# es un match de palabra clave y no mide nada sobre atención en contexto largo. Con 20
# señuelos del mismo formato, el modelo tiene que DISCRIMINAR cuál corresponde al
# laboratorio — que es el problema que Lost in the Middle dice que se degrada con la
# posición. (Versiones anteriores de este notebook daban OK en todas las celdas
# justamente porque la tarea era demasiado fácil; esto es un intento de romperla.)
SENUELOS = ["VERDE-17", "ROJO-08", "GRIS-23", "OCRE-55", "AZUL-11", "MALVA-72",
            "CIAN-30", "SEPIA-46", "LILA-64", "CORAL-19", "JADE-88", "ÁMBAR-05",
            "TOPACIO-91", "ÍNDIGO-37", "BEIGE-26", "CARMÍN-73", "TURQUESA-42",
            "OLIVA-58", "PÚRPURA-14", "SALMÓN-69"]

def distractor(i, t):
    return ("Nota " + str(i) + ". Sobre " + t + ": es un tema de tradición larga y "
            "bibliografía dispersa, que suele estudiarse por regiones antes que de forma "
            "global. El código de referencia del archivo temático es " + SENUELOS[i] + ", "
            "asignado por la comisión de catalogación en su última revisión. Quienes lo "
            "investigan discuten sobre todo cuestiones de método y de fuentes, y rara vez "
            "llegan a conclusiones unánimes. Los relevamientos más citados son de hace "
            "décadas y hoy se los considera incompletos.")

DISTRACTORES = [distractor(i, t) for i, t in enumerate(temas)]
NEEDLE = "El código de acceso del laboratorio es AZUL-42."
PREGUNTA = ("¿Cuál es el código de acceso del LABORATORIO? (No el código de referencia de "
            "ningún archivo temático.) Respondé solo el código.")

def armar_contexto(n_dist, pos_frac):
    ds = DISTRACTORES[:n_dist]
    k = int(pos_frac * len(ds))
    lineas = ds[:k] + [NEEDLE] + ds[k:]        # el needle queda en la posición k
    return "\n".join(lineas), k, len(lineas)

def probar(n_dist, pos_frac, modelo):
    contexto, _, _ = armar_contexto(n_dist, pos_frac)
    out = ask(contexto + "\n\n" + PREGUNTA, model=modelo, num_predict=32, avisar=False)
    return "azul-42" in out.lower()

contexto, k, total = armar_contexto(12, 0.5)
lineas = contexto.split("\n")
corta = lambda s: s[:110] + (" ..." if len(s) > 110 else "")
antes, despues = max(0, k - 2), max(0, total - k - 1)   # max(0,...): con pos_frac=0.0, k=0
elidido = "\n".join(
    [corta(l) for l in lineas[:2]]
    + ["   ... (" + str(antes) + " distractores más, cada uno con SU código) ..."]
    + [">>> " + lineas[k] + "   <-- el needle, en la posición " + str(k) + " de " + str(total)]
    + ["   ... (" + str(despues) + " distractores más, cada uno con SU código) ..."])
print("Contexto de 12 distractores ≈", len(contexto.split()), "palabras |",
      "códigos señuelo en el contexto:", 12, "\n")
mostrar_ejemplo("LOST IN THE MIDDLE — prompt (elidido) + respuesta",
                elidido + "\n\n" + PREGUNTA,
                ask(contexto + "\n\n" + PREGUNTA, num_predict=32, avisar=False))

POSICIONES = [("inicio", 0.0), ("medio", 0.5), ("final", 1.0)]
filas, res = [], {}
for nombre, modelo in ESCALAS:
    for n in [12, 20]:
        fila = {pos: ("OK" if probar(n, frac, modelo) else "x") for pos, frac in POSICIONES}
        filas.append((nombre + " / " + str(n) + " dist.", fila))
        res[nombre + "_" + str(n)] = fila

tabla("¿Encuentra el needle entre los señuelos? (por escala y largo de contexto)",
      [p for p, _ in POSICIONES], filas)

RESULTADOS["lost_in_the_middle"] = res

Contexto de 12 distractores ≈ 942 palabras | códigos señuelo en el contexto: 12 

──────────────────────────────────────────────────────────────────────
LOST IN THE MIDDLE — prompt (elidido) + respuesta
──────────────────────────────────────────────────────────────────────
PROMPT:
Nota 0. Sobre el clima de la Patagonia: es un tema de tradición larga y bibliografía dispersa, que suele estud ...
Nota 1. Sobre la historia del café: es un tema de tradición larga y bibliografía dispersa, que suele estudiars ...
   ... (4 distractores más, cada uno con SU código) ...
>>> El código de acceso del laboratorio es AZUL-42.   <-- el needle, en la posición 6 de 13
   ... (6 distractores más, cada uno con SU código) ...

¿Cuál es el código de acceso del LABORATORIO? (No el código de referencia de ningún archivo temático.) Respondé solo el código.

RESPUESTA DEL MODELO:
AZUL-42


¿Encuentra el needle entre los señuelos? (por escala y largo de contexto)
────────────────────────────────────────────────

**Cómo leerlo.** Cuatro filas: dos escalas × dos largos de contexto. Y una advertencia de honestidad que es, en sí misma, parte de la clase.

Si el "medio" falla más que "inicio"/"final", viste la **curva en U** (Lost in the Middle). Si la fila de 20 distractores anda peor que la de 12 con el mismo modelo, viste **Context Rot**. Y si el modelo chico falla donde el 4B acierta, viste que la robustez al contexto largo también es capacidad.

> **Si te dio todo OK, eso también es un dato — y es el resultado más probable.** Esta demo se resistió a varios intentos de endurecerla: distractores largos, veinte de ellos, un needle que no grita, y un código señuelo en cada distractor para que encontrarlo no sea un match de palabra clave. Aun así, un modelo de 0,5B suele recuperar el dato desde cualquier posición.
>
> La razón importa más que el resultado: **recuperar una línea literal es de las cosas más fáciles que hace la atención.** Los efectos de Lost in the Middle y Context Rot se miden con contextos de decenas de miles de tokens, sobre tareas que exigen *combinar* información dispersa —no encontrar una frase— y promediando muchas corridas. Chroma lo hizo sobre 18 modelos, Qwen3 incluido. Nosotros tenemos ~1.000 palabras y una sola corrida por celda: una lupa de juguete para un fenómeno estadístico.
>
> Quedate con dos cosas. La primera, el **método**: variar la posición de un dato manteniendo todo lo demás constante es la forma correcta de aislar el efecto de la posición, y es reutilizable en cualquier sistema que armes. La segunda, la **disciplina**: una demo que no reproduce el efecto que anuncia no se tapa ni se reescribe el texto para que parezca que sí. Se dice, se explica por qué, y se marca qué haría falta para verlo de verdad.

> **La consecuencia práctica (que no depende de esta medición):** el contexto es un recurso finito con rendimientos decrecientes. No se trata de meter *más*, sino de curar *lo justo*. Eso es lo que viene.

▶ **Lámina 34** (Lost in the Middle + Context Rot; el presupuesto de atención).


## 16 · El nivel meta: medir para mejorar, y automatizarlo (APE)

Cerramos volviendo al instrumento. Un prompt mediocre, medido; lo mejoramos a mano; volvemos a medir. Después lo **automatizamos**: el modelo **propone** variantes, el harness las **puntúa**, nos quedamos con la mejor. Eso es APE en miniatura — el prompt como algo que se **busca**, no que se adivina.

Pero medimos con dos instrumentos, no uno. La tarea pide un tweet de **≤280 caracteres**, y eso no se opina: se cuenta. Así que al lado del score del juez corre un **verificador programático** de longitud, y el ranking final usa el producto de los dos.

> **Mirá con atención el largo de cada tweet contra su score.** Acá está el experimento más importante de la sección, y no es sobre APE.


In [47]:
LIMITE = 280
tarea = "Escribir un tweet claro y atractivo que explique un concepto en <=280 caracteres."
dataset_meta = [{"concepto": "qué es el interés compuesto"},
                {"concepto": "qué es una VPN"},
                {"concepto": "qué es el machine learning"}]
crit_meta = "Correcto, claro, con un gancho, y en un solo tweet (<=280 caracteres)."

def evaluar_prompt(instruccion):
    """Devuelve (avg_juez, frac_cumple_limite, puntaje_compuesto, resultados)."""
    def run(inp):
        return ask(instruccion + "\nConcepto: " + inp["concepto"], num_predict=400)
    r = evaluator.run_evaluation(run, dataset_meta, task_description=tarea,
                                 extra_criteria=crit_meta)
    avg = sum(x["score"] for x in r) / len(r)
    cumple = sum(1 for x in r if len(x["output"]) <= LIMITE) / len(r)
    # Métrica compuesta: el juez opina sobre la calidad, pero el límite de caracteres NO
    # se opina, se mide. Un tweet que no entra en un tweet vale cero por bueno que suene.
    return round(avg, 2), round(cumple, 2), round(avg * cumple, 2), r

base     = "Escribí un tweet sobre el concepto."
mejorado = ("Sos community manager. Escribí UN tweet (<=280 caracteres) que explique el "
            "concepto con una analogía cotidiana y un gancho en la primera línea. Sin hashtags.")

avg_base, cum_base, comp_base, r_base = evaluar_prompt(base)
avg_mej,  cum_mej,  comp_mej,  r_mej  = evaluar_prompt(mejorado)

c0 = dataset_meta[0]["concepto"]
mostrar_ejemplo("PROMPT BASE — prompt + tweet", base + "\nConcepto: " + c0, r_base[0]["output"])
print("   juez:", r_base[0]["score"], "/10  |  largo real:", len(r_base[0]["output"]),
      "caracteres ->", "CUMPLE" if len(r_base[0]["output"]) <= LIMITE else "VIOLA el límite de 280")
mostrar_ejemplo("PROMPT MEJORADO — prompt + tweet", mejorado + "\nConcepto: " + c0, r_mej[0]["output"])
print("   juez:", r_mej[0]["score"], "/10  |  largo real:", len(r_mej[0]["output"]),
      "caracteres ->", "CUMPLE" if len(r_mej[0]["output"]) <= LIMITE else "VIOLA el límite de 280")

# Caso por caso: el juez al lado del verificador duro.
print("\n" + "concepto".ljust(30), "juez", "  chars", "  <=280?")
for etiqueta, res in [("BASE", r_base), ("MEJOR", r_mej)]:
    for x in res:
        n = len(x["output"])
        print((etiqueta + " " + x["inputs"]["concepto"])[:28].ljust(30),
              str(x["score"]).rjust(4), str(n).rjust(7), "  ", "sí" if n <= LIMITE else "NO")

tabla("Prompt base vs mejorado",
      ["juez (0-10)", "cumple 280", "compuesto"],
      [("base",     {"juez (0-10)": avg_base, "cumple 280": cum_base, "compuesto": comp_base}),
       ("mejorado", {"juez (0-10)": avg_mej,  "cumple 280": cum_mej,  "compuesto": comp_mej})])

RESULTADOS["meta"] = {"juez base": avg_base, "juez mejorado": avg_mej,
                      "cumple 280 base": cum_base, "cumple 280 mejorado": cum_mej}

──────────────────────────────────────────────────────────────────────
PROMPT BASE — prompt + tweet
──────────────────────────────────────────────────────────────────────
PROMPT:
Escribí un tweet sobre el concepto.
Concepto: qué es el interés compuesto

RESPUESTA DEL MODELO:
¡El interés compuesto! 📈  
Es cuando el interés que ganas no solo se suma al capital, sino que también **genera más interés**.  
Como si cada año, tu dinero empezara a "ganar dinero por sí mismo".  
Ejemplo: $100 a 5% al año → al año 1: $105, al año 2: $110.25… y así, crece exponencialmente.  
¡La clave del crecimiento sostenido! 💡  
#Finanzas #InterésCompuesto

   juez: 10 /10  |  largo real: 363 caracteres -> VIOLA el límite de 280
──────────────────────────────────────────────────────────────────────
PROMPT MEJORADO — prompt + tweet
──────────────────────────────────────────────────────────────────────
PROMPT:
Sos community manager. Escribí UN tweet (<=280 caracteres) que explique el concepto con una analogía co

In [48]:
# mini-APE: el modelo propone instrucciones, el harness las puntúa, elegimos la mejor.
# Los candidatos salen con STRUCTURED OUTPUTS, no partiendo el texto por líneas: partir
# por líneas metía el preámbulo del modelo ("Claro, acá van tres:") como candidato 1.
class Candidatos(BaseModel):
    instrucciones: list[str]

# Al proponente le pasamos la tarea COMPLETA, restricciones incluidas. Es lo que hace un
# APE de verdad: si el que propone no conoce la métrica, explora un espacio equivocado y
# la búsqueda no puede encontrar nada bueno por más candidatos que genere.
raw_ape = ask("Proponé " + str(N_APE) + " instrucciones DISTINTAS para resolver esta tarea.\n"
              "TAREA: " + tarea + "\n"
              "CRITERIOS con los que se van a evaluar: " + crit_meta + "\n"
              "Cada instrucción es el texto que va a recibir OTRO modelo para escribir el "
              "tweet. Tienen que mencionar explícitamente las restricciones de la tarea.",
              temperature=1.0, fmt=Candidatos.model_json_schema())
candidatos = Candidatos.model_validate_json(raw_ape).instrucciones[:N_APE]

print("Candidatos propuestos:")
for c in candidatos:
    print("  -", c[:100])

# Los puntuamos con la MISMA métrica compuesta de arriba: juez × cumplimiento del límite.
# Incluimos los dos prompts escritos a mano para ver si el APE les gana de verdad.
print("\nEvaluando...")
ranking = []
for etiqueta, c in [("(a mano) base", base), ("(a mano) mejorado", mejorado)] + \
                   [("APE " + str(i + 1), c) for i, c in enumerate(candidatos)]:
    avg, cum, comp, _ = evaluar_prompt(c)
    ranking.append((comp, avg, cum, etiqueta, c))
ranking.sort(reverse=True)

print("\n" + "prompt".ljust(20), "compuesto", " juez", " cumple280")
for comp, avg, cum, etiqueta, c in ranking:
    print(etiqueta.ljust(20), str(comp).rjust(9), str(avg).rjust(5), str(cum).rjust(10))

print("\nGanador:", ranking[0][3], "->", ranking[0][4][:90])
RESULTADOS["ape"] = {"ganador": ranking[0][3], "compuesto": ranking[0][0]}

Candidatos propuestos:
  - Escribe un tweet claro y atractivo que explique un concepto en máximo 280 caracteres. Debe ser corre
  - Crea un tweet breve (máximo 280 caracteres) que defina un concepto de forma clara, precisa y con un 

Evaluando...

prompt               compuesto  juez  cumple280
APE 2                     10.0  10.0        1.0
APE 1                     10.0  10.0        1.0
(a mano) mejorado         7.67  7.67        1.0
(a mano) base              0.0  9.33        0.0

Ganador: APE 2 -> Crea un tweet breve (máximo 280 caracteres) que defina un concepto de forma clara, precisa


**Tres cierres: uno técnico, uno incómodo y uno sobre la búsqueda misma.**

*El técnico.* El prompt dejó de ser artesanía y pasó a ser un **objeto optimizable**: definís una métrica, generás candidatos, medís, seleccionás. Es el mismo loop de todo el aprendizaje automático, ahora sobre el prompt. Y descansa entero sobre el instrumento de la Sección 2 — sin métrica no hay búsqueda.

*El incómodo.* Si algún tweet se pasó de 280 caracteres y **aun así el juez le puso un puntaje alto**, mirálo bien: el criterio decía "≤280 caracteres" en el prompt del juez, el juez lo leyó, y lo aprobó igual. Un LLM no cuenta caracteres — predice tokens. Pedirle que verifique una restricción aritmética es pedirle exactamente lo que no sabe hacer. Eso es la Sección 13(c) con un costo concreto: **verificable → verificador duro; subjetivo → juez.** Confundirlos no da un error, da un número que parece bueno.

*El de la búsqueda.* Mirá el ranking completo, incluidos los dos prompts escritos a mano. **Si ganó uno de los humanos, el APE no fracasó: te mostró su límite real.** Una búsqueda sólo puede encontrar lo que sus candidatos exploran, y el que propone candidatos es el mismo tipo de modelo, con los mismos puntos ciegos. Si el proponente no tiene presente la restricción que la métrica castiga, va a generar variaciones de *estilo* —más gancho, más metáfora, más emoji— y ninguna va a mover el número que importa.

Por eso acá al proponente le pasamos la tarea **con sus criterios de evaluación adentro**. Es la diferencia entre "generá prompts lindos" y "generá prompts que maximicen esta métrica", y es donde vive la mitad del trabajo de un APE real. La otra mitad es la métrica: **una búsqueda no puede ser mejor que la función que la guía.** Si tu métrica premia lo equivocado, la optimización te va a llevar ahí con una eficiencia notable.

▶ **Láminas 31–33** (el prompt como código; APE; el prompt como programa que se busca).


## 17 · Cierre: lo que medimos, y qué sigue

Todo lo que vimos son operaciones sobre **una** distribución del próximo token, sobre pesos congelados. La tabla de abajo **no está escrita a mano: la genera la corrida** con los números que acabás de producir. Si un efecto no aparece, aparece como no-efecto — que es justamente lo que queremos que se vea.

| Técnica | Operación sobre la distribución | Láminas |
|---|---|---|
| Few-shot | condicionar con ejemplos | 4–6 |
| Instrucciones claras | condicionar con precisión | 7 |
| Chain-of-Thought | expandir el cómputo | 10–12 |
| Zero-shot CoT / RE2 | expandir (gatillo / input) | 13–14 |
| Self-consistency | muestrear y votar | 15 |
| `think=True` | capacidad horneada por RL | 17 |
| Structured outputs | restringir (máscara de logits) | 19–26 |
| Nombres/orden de campos | keys = tokens de salida | 27 |
| Function calling | restringir hacia una acción | 29–30 |
| Context (posición/largo) | curar el input | 34 |
| APE / meta | buscar el prompt | 31–33 |

**El backbone.** Detrás de las dos palancas —expandir y restringir— está la pregunta de la Clase 6: cuándo movés la **forma** y cuándo la **capacidad**. El prompt mueve forma sobre una capacidad ya entrenada; por eso ninguna de estas técnicas le enseña nada nuevo al modelo — a lo sumo lo ayudan a **explotar mejor** lo que ya tiene.

**Cuatro cosas que aprendimos midiendo, y que ningún deck dice.**

1. **Toda técnica tiene un rango de operación.** Arriba del rango sobra (el 4B empata en casi todo); abajo del rango puede hacer daño (el colapso de few-shot en el modelo diminuto). "¿Funciona?" es la pregunta mal hecha; la pregunta es "¿en qué régimen?".
2. **El juez mejora cuando la rúbrica es verificable** (Sección 4) y **falla cuando el criterio es aritmético** (Sección 16: aprobó tweets que se pasaban del límite de caracteres que él mismo tenía en la rúbrica). Lo que se puede contar, contalo con código.
3. **La garantía de structured outputs es sobre la gramática, no sobre que el modelo termine.** Un JSON perfectamente válido token a token puede quedar sin cerrar si se acaba el presupuesto.
4. **Los detalles de implementación mandan.** El backend ordena las claves alfabéticamente, y eso decide qué se genera primero. Ningún paper te lo va a decir: lo ves mirando la salida cruda.

**Lo que sigue.** Vimos cómo *dirigir* lo que el modelo ya sabe. Pero cuando el conocimiento **no está** en los pesos —documentos privados, hechos posteriores al entrenamiento— condicionar no alcanza: hay que **traer** la información al contexto y recuperarla bien (justo el problema de posición y largo que acabamos de medir). Eso es **RAG y embeddings — Unidad 4**.


In [49]:
# La tabla de resultados se GENERA desde lo que midió la corrida. Si una sección no se
# corrió, aparece un guión en vez de una afirmación inventada.
def fmt(v, sangria="   "):
    if isinstance(v, dict):
        if any(isinstance(x, dict) for x in v.values()):      # dict anidado (por escala)
            return "\n".join(sangria + str(k).ljust(14) + fmt(x, "")
                             for k, x in v.items())
        return sangria + "   ".join(str(k) + "=" + str(x) for k, x in v.items())
    return sangria + str(v)

FILAS = [
    ("Few-shot (accuracy)",            "few_shot"),
    ("Instrucciones claras",           "claro_vs_vago"),
    ("Chain-of-Thought",               "cot"),
    ("Zero-shot CoT / RE2",            "zscot_re2"),
    ("Self-consistency",               "self_consistency"),
    ("Razonamiento nativo",            "razonador"),
    ("JSON a mano vs format=",         "json"),
    ("Orden de campos (accuracy)",     "orden_campos"),
    ("Contexto: posición y largo",     "lost_in_the_middle"),
    ("APE (métrica compuesta)",        "ape"),
]

print("=" * 78)
print("LO QUE MIDIÓ ESTA CORRIDA".center(78))
print("=" * 78)
for etiqueta, clave in FILAS:
    v = RESULTADOS.get(clave)
    print("\n" + etiqueta)
    print(fmt(v) if v is not None else "   — (no se corrió esta sección)")

cadenas = RESULTADOS.get("self_consistency_cadenas")
if cadenas:
    print("\n" + "-" * 78)
    print("Recordatorio de por qué el voto ayuda o no — cadenas sobre el mismo problema:")
    for k, v in cadenas.items():
        print("   " + k.ljust(7), v, "->", len(set(v)), "respuesta(s) distinta(s)")
print("\n" + "=" * 78)

                          LO QUE MIDIÓ ESTA CORRIDA                           

Few-shot (accuracy)
   diminuto      zero=0.67   one=0.33   few-OK=0.33   few-RND=0.67
   chico         zero=0.83   one=0.92   few-OK=0.92   few-RND=1.0
   4B            zero=0.92   one=0.92   few-OK=1.0   few-RND=1.0

Instrucciones claras
   juez vago=2.0   juez claro=5.75   formato vago=0/4   formato claro=4/4

Chain-of-Thought
   directo=0.17   CoT=0.83   palabras directo=1.0   palabras CoT=87.8

Zero-shot CoT / RE2
   zero-shot CoT=0.83   RE2+CoT=1.0

Self-consistency
   diminuto      1 muestra=0.0   voto de 5=0.17   sin mayoría=0/6
   4B            1 muestra=0.83   voto de 5=1.0   sin mayoría=0/6

Razonamiento nativo
   instruct directo=0.17   think=True=1.0

JSON a mano vs format=
   a mano 4B (json.loads)=3/3   a mano chico (claves+tipos)=1/3   con format= 4B=3/3   con format= chico=3/3

Orden de campos (accuracy)
   diminuto      razonamiento primero=0.67   respuesta primero=0.83
   4B            ra